# TRUEGUARD — Official IEEE Access 10-Dataset Experimental Pipeline (`FINAL`)
**Paper Title:** *TRUEGUARD: A White-Box Framework for LLM Hallucination Detection and Risk-Based Mitigation*

### Self-Contained Design (No Manual ZIP Upload Required)
1. **Exact Original Architecture Embedded:** This notebook writes the verified original `src/tg_pipeline.py`, `src/tg_controller.py`, and `src/tg_eval.py` directly into the Colab runtime — zero missing modules, zero Windows paths, zero ZIP uploads.
2. **Frozen Foundation Model:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0` loaded with `attn_implementation="eager"` and 100% frozen weights (`trainable_params == 0`).
3. **All Four Original White-Box Sensors:**
   - **ISP (Internal State Probe):** Per-layer logistic probes trained on answer-mean hidden states at depth fractions `(0.45, 0.60, 0.75, 0.90)` with grouped inner-OOF regularization selection (zero probe-to-fusion leakage).
   - **AAS (Attention Anomaly Score):** Normalized Shannon entropy of causal attention rows across the final quartile of transformer layers.
   - **EEM (Energy-Entropy Monitor):** `0.6 * H_shannon_norm + 0.4 * H_renyi2_norm` of output logits.
   - **DST (Directional Semantic Trajectory / Distribution Shift Tracker):** `(1 - cos(h_t, h_{t-1})) / 2` on layer-averaged hidden states at depth fractions `(0.60, 0.75, 0.90)`.
   - **4-Sensor Fusion:** Logistic regression meta-classifier trained on inner-OOF ISP log-odds + AAS + EEM + DST.
4. **10-Dataset Methodology (Parquet-Only, Zero Unsupported HF Scripts):**
   - **In-Domain Training & 5-Fold GroupKFold CV (5 datasets):** `halueval_dialogue` (2,000 pairs = 4,000 seq), `halueval_qa` (2,000 pairs = 4,000 seq), `halueval_summarization` (2,000 pairs = 4,000 seq), `truthful_qa` (all 817 pairs = 1,634 seq), `halueval_general` (all 4,507 responses).
   - **Strictly Unseen Zero-Shot Evaluation (4 datasets — 0 samples in training/scaling/calibration/thresholding):** `faithdial` (all 3,507 grounded turns across 791 dialogs), `wikibio_gpt3` (all 1,908 sentences across 238 articles), `fever` (all 6,666 unique claims — explicitly framed as *claim-verification / truthfulness transfer*), `xsum_factuality` (all 1,868 human-consensus summaries joined with real BBC source articles).
   - **Risk-Based Mitigation (1 dataset):** `trivia_qa` (`n_cal=250`, `n_test=250` disjoint closed-book questions) with full `ThreeTierController` (`SAFE`, `TIER1_RESAMPLE`, `TIER2_RETRIEVAL` via Wikipedia, `TIER3_ABSTAIN`) and selective risk–coverage analysis.
5. **Resilient Google Drive Checkpointing:** Saves incremental feature batches every 100 samples with SHA-256 dataset & code-version verification so interrupted Colab runs resume cleanly without reusing stale or incompatible caches.


In [ ]:
# STEP 1: Install and Verify Dependencies (no unsupported HF dataset scripts used)
!pip install -q "transformers>=4.44.0" "accelerate>=0.30.0" "huggingface_hub>=0.24.0" "pyarrow>=14.0.0" "pandas>=2.0.0" "scikit-learn>=1.3.0" "scipy>=1.11.0" "matplotlib>=3.7.0" "tabulate>=0.9.0" "joblib>=1.3.0" "requests>=2.31.0"

import sys, platform, torch, transformers, huggingface_hub, sklearn, numpy as np, pandas as pd, pyarrow
print("=== ENVIRONMENT VERIFICATION ===")
print(f"Python:          {sys.version.split()[0]} ({platform.system()})")
print(f"PyTorch:         {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")
else:
    print("WARNING: No CUDA GPU detected! Select Runtime -> Change runtime type -> T4 GPU.")
print(f"Transformers:    {transformers.__version__}")
print(f"HF Hub:          {huggingface_hub.__version__}")
print(f"scikit-learn:    {sklearn.__version__} | pandas: {pd.__version__} | pyarrow: {pyarrow.__version__}")


In [ ]:
# STEP 2: Mount Google Drive & Create Isolated Output Directories
# Uses a dedicated folder TRUEGUARD_IEEE_FINAL so all previous runs/checkpoints remain untouched.
import os, sys, json, time, random

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = "/content/drive/MyDrive/TRUEGUARD_IEEE_FINAL"
else:
    RUN_ROOT = os.path.abspath("TRUEGUARD_IEEE_FINAL_LOCAL")

DIRS = {
    "root":        RUN_ROOT,
    "checkpoints": os.path.join(RUN_ROOT, "checkpoints"),
    "models":      os.path.join(RUN_ROOT, "models"),
    "predictions": os.path.join(RUN_ROOT, "predictions"),
    "figures":     os.path.join(RUN_ROOT, "figures"),
    "results":     os.path.join(RUN_ROOT, "results"),
}
for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

print("=== PERSISTENT STORAGE READY ===")
for k, v in DIRS.items():
    print(f"  {k:12s}: {v}")


In [ ]:
# STEP 3: Reproducibility Settings & Master Experimental Configuration
SEED = 42
PIPELINE_VERSION = "TRUEGUARD_v2_IEEE_FINAL_2026_11"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# Explicit dataset roles & sample caps:
#   - None = 100% of all valid rows in the benchmark.
#   - HaluEval Dialogue / QA / Summarization each have 10,000 raw pairs (20,000 sequences each = 60,000 total).
#     We use 2,000 pairs (4,000 sequences) each — 5x the original 400-pair study — giving 95% CI width < +/-0.008
#     while keeping total T4 GPU extraction time to ~1.5-2.5 hours.
IN_DOMAIN_DATASETS = [
    "halueval_dialogue",
    "halueval_qa",
    "halueval_summarization",
    "truthful_qa",
    "halueval_general",
]
UNSEEN_ZEROSHOT_DATASETS = [
    "faithdial",
    "wikibio_gpt3",
    "fever",             # Explicitly analyzed as claim-verification / truthfulness-transfer benchmark
    "xsum_factuality",
]
DATASET_CAPS = {
    "halueval_dialogue":      2000,   # pairs (-> 4,000 sequences)
    "halueval_qa":            2000,   # pairs (-> 4,000 sequences)
    "halueval_summarization": 2000,   # pairs (-> 4,000 sequences)
    "truthful_qa":            None,   # 100% (817 pairs -> 1,634 sequences)
    "halueval_general":       None,   # 100% (4,507 responses)
    "faithdial":              None,   # 100% valid grounded turns (3,507 turns across 791 dialogs)
    "wikibio_gpt3":           None,   # 100% (1,908 sentences across 238 articles)
    "fever":                  None,   # 100% deduplicated SUPPORTS/REFUTES (6,666 unique claims)
    "xsum_factuality":        None,   # 100% non-tie consensus summaries (1,868 summaries across 498 articles)
}
TRIVIAQA_N_CAL  = 250   # Strictly disjoint calibration split for ThreeTierController
TRIVIAQA_N_TEST = 250   # Strictly disjoint test split for ThreeTierController
USE_WIKIPEDIA_RETRIEVAL = True
LATENCY_N_PROMPTS = 25
LATENCY_N_TOKENS  = 24
N_BOOT = 500
CHECKPOINT_BATCH = 100

# Original verified TRUEGUARD v2 TinyLlama reference metrics (for historical comparison ONLY, NEVER substituted for new runs)
ORIGINAL_V2_REFERENCE = {
    "halueval_dialogue":      {"n_rows": 800, "fusion_auroc": 0.9028, "isp_auroc": 0.9039, "f1": 0.820, "ece": 0.034},
    "halueval_qa":            {"n_rows": 800, "fusion_auroc": 0.9749, "isp_auroc": 0.9771, "f1": 0.955, "ece": 0.024},
    "halueval_summarization": {"n_rows": 800, "fusion_auroc": 0.9863, "isp_auroc": 0.9867, "f1": 0.935, "ece": 0.025},
    "truthful_qa":            {"n_rows": 796, "fusion_auroc": 0.8763, "isp_auroc": 0.8772, "f1": 0.817, "ece": 0.042},
    "trivia_qa_mitigation":   {"n_cal": 150, "n_test": 150, "test_auroc": 0.7948, "before_halluc": 0.620, "after_answered_halluc": 0.400, "after_system_halluc": 0.173, "coverage": 0.433},
}

AUDIT = {"pipeline_version": PIPELINE_VERSION, "model": MODEL_NAME, "seed": SEED, "stages": {}, "started_at": time.strftime("%Y-%m-%d %H:%M:%S")}
print(f"Configuration locked: {PIPELINE_VERSION} | Model: {MODEL_NAME}")


In [ ]:
# STEP 4: Materialize Original TRUEGUARD v2 Source Modules (Zero ZIP Upload Needed)
# Writes the exact verified src/tg_pipeline.py, src/tg_controller.py, and src/tg_eval.py to ./src/
import base64, hashlib

os.makedirs("src", exist_ok=True)
with open("src/__init__.py", "w", encoding="utf-8") as f:
    f.write("")

_EMBEDDED_SRC = {
    "tg_pipeline.py":   "...",
    "tg_controller.py": "IiIiClRSVUVHVUFSRCB2MiB0aHJlZS10aWVyIGNsb3NlZC1sb29wIGNvbnRyb2xsZXIgKHJldmlldyBwYXBlciBTZWN0aW9uIDYuMy02LjQpLgoKUmVnaW1lcyAodGhyZXNob2xkcyBhcmUgY2FsaWJyYXRlZCBvbiBoZWxkLW91dCB2YWxpZGF0aW9uIGRhdGEsIG5ldmVyIG9uIHRlc3QgZGF0YSk6CiAgICByaXNrIDwgIHRhdV93YXJuaW5nICAgICAgICAgICAgICAgICAgU0FGRSAgICAgICAgICAgIC0+IHJlbGVhc2UgYW5zd2VyIHVuY2hhbmdlZAogICAgdGF1X3dhcm5pbmcgPD0gcmlzayA8IHRhdV9mYXVsdCAgICAgIFdBUk5JTkcgICAgICAgICAtPiBUaWVyIDE6IHJpc2stZ3VpZGVkIGNvcnJlY3RpdmUgcmUtZ2VuZXJhdGlvbgogICAgcmlzayA+PSB0YXVfZmF1bHQgICAgICAgICAgICAgICAgICAgIEZBVUxUICAgICAgICAgICAtPiBUaWVyIDI6IHJldHJpZXZhbC1ncm91bmRlZCByZS1nZW5lcmF0aW9uLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBUaWVyIDM6IGNhbGlicmF0ZWQgYWJzdGVudGlvbiBpZiBzdGlsbCByaXNreQpFc2NhbGF0aW9uOiBpZiBUaWVyIDEgY2Fubm90IGZpbmQgYSBjYW5kaWRhdGUgYmVsb3cgdGF1X3dhcm5pbmcgdGhlIGNvbnRyb2xsZXIgZXNjYWxhdGVzIHRvIFRpZXIgMjsgaWYgdGhlCnJldHJpZXZhbC1ncm91bmRlZCBhbnN3ZXIgaXMgc3RpbGwgPj0gdGF1X2ZhdWx0IChvciBub3RoaW5nIGlzIHJldHJpZXZlZCkgaXQgYWJzdGFpbnMgKFRpZXIgMykuCgpEb2N1bWVudGVkIGRldmlhdGlvbiBmcm9tIHRoZSBwYXBlcjogVGllciAxIHJlLXNhbXBsZXMgdGhlIHdob2xlIChzaG9ydCkgYW5zd2VyIHJhdGhlciB0aGFuIHJvbGxpbmcgYmFjayAiYSBmZXcKdG9rZW5zIiwgYmVjYXVzZSB0aGUgZGV0ZWN0b3Igc2NvcmVzIGNvbXBsZXRlIGFuc3dlcnM7IGZvciBhbnN3ZXJzIG9mIDw9IDMyIHRva2VucyB0aGlzIGlzIGEgZmV3IHRva2VucyBhbnl3YXkuCkl0IGRyYXdzIEsgY2FuZGlkYXRlIGFuc3dlcnMgKHRlbXBlcmF0dXJlIHNhbXBsaW5nKSBhbmQga2VlcHMgdGhlIGxvd2VzdC1yaXNrIG9uZSAtLSBhIHJlYWwgY29ycmVjdGl2ZSByZWdlbmVyYXRpb24KZHJpdmVuIGJ5IHRoZSBkZXRlY3Rvciwgbm90IGEgcGxhY2Vob2xkZXIuCiIiIgppbXBvcnQganNvbgppbXBvcnQgdGltZQppbXBvcnQgdXJsbGliLnBhcnNlCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLnRnX3BpcGVsaW5lIGltcG9ydCBnZW5lcmF0ZV9hbnN3ZXIsIHNjb3JlX2Fuc3dlciwgYnVpbGRfcHJvbXB0CgpBQlNUQUlOX1RFWFQgPSAiSSBkbyBub3QgaGF2ZSBzdWZmaWNpZW50IHZlcmlmaWVkIGV2aWRlbmNlIHRvIGFuc3dlciB0aGlzIHJlbGlhYmx5LiIKCgpjbGFzcyBXaWtpcGVkaWFSZXRyaWV2ZXI6CiAgICAiIiJUYXJnZXRlZCByZXRyaWV2YWwgb2YgYW4gZXZpZGVuY2UgcGFzc2FnZSBmb3IgdGhlIHF1ZXN0aW9uIChUaWVyIDIpLiAgTmVlZHMgaW50ZXJuZXQuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1heF9jaGFycz03MDAsIHRpbWVvdXQ9Ni4wKToKICAgICAgICBzZWxmLm1heF9jaGFycywgc2VsZi50aW1lb3V0LCBzZWxmLmNhY2hlID0gbWF4X2NoYXJzLCB0aW1lb3V0LCB7fQoKICAgIGRlZiBfX2NhbGxfXyhzZWxmLCBxdWVyeSk6CiAgICAgICAgaWYgIlF1ZXN0aW9uOiIgaW4gcXVlcnk6CiAgICAgICAgICAgIHF1ZXJ5ID0gcXVlcnkuc3BsaXQoIlF1ZXN0aW9uOiIsIDEpWzFdLnN0cmlwKCkKICAgICAgICBxdWVyeSA9IHF1ZXJ5LnNwbGl0bGluZXMoKVswXS5zdHJpcCgpLnN0cmlwKCciPycpCiAgICAgICAgc3RvcCA9IHsid2hhdCIsICJ3aGljaCIsICJ3aG8iLCAid2hvbSIsICJ3aG9zZSIsICJ3aGVuIiwgIndoZXJlIiwgIndoeSIsICJob3ciLCAiaXMiLCAid2FzIiwgImFyZSIsICJ3ZXJlIiwKICAgICAgICAgICAgICAgICJkaWQiLCAiZG9lcyIsICJkbyIsICJoYXMiLCAiaGF2ZSIsICJoYWQiLCAidGhlIiwgImEiLCAiYW4iLCAiaW4iLCAib24iLCAib2YiLCAidG8iLCAiZm9yIiwgIndpdGgiLAogICAgICAgICAgICAgICAgImJ5IiwgImZyb20iLCAiYXMiLCAiYXQiLCAiaW50byIsICJjYWxsZWQiLCAia25vd24iLCAibmFtZSIsICJuYW1lZCIsICJ3cm90ZSIsICJtYWRlIiwgInBsYXllZCJ9CiAgICAgICAgd29yZHMgPSBbdy5zdHJpcCgiLC4/XCInKCkiKSBmb3IgdyBpbiBxdWVyeS5zcGxpdCgpXQogICAgICAgIGtlcHQgPSBbdyBmb3IgdyBpbiB3b3JkcyBpZiB3IGFuZCB3Lmxvd2VyKCkgbm90IGluIHN0b3BdCiAgICAgICAgaWYga2VwdDoKICAgICAgICAgICAgcXVlcnkgPSAiICIuam9pbihrZXB0Wzo4XSkKICAgICAgICBpZiBxdWVyeSBpbiBzZWxmLmNhY2hlOgogICAgICAgICAgICByZXR1cm4gc2VsZi5jYWNoZVtxdWVyeV0KICAgICAgICBwYXNzYWdlID0gTm9uZQogICAgICAgIHRyeToKICAgICAgICAgICAgaW1wb3J0IHJlcXVlc3RzCiAgICAgICAgICAgIGggPSB7IlVzZXItQWdlbnQiOiAiVFJVRUdVQVJELVJlc2VhcmNoLzIuMCAoYWNhZGVtaWMgZXZhbHVhdGlvbikifQogICAgICAgICAgICByID0gcmVxdWVzdHMuZ2V0KCJodHRwczovL2VuLndpa2lwZWRpYS5vcmcvdy9hcGkucGhwIiwgaGVhZGVycz1oLCB0aW1lb3V0PXNlbGYudGltZW91dCwgcGFyYW1zPWRpY3QoCiAgICAgICAgICAgICAgICBhY3Rpb249InF1ZXJ5IiwgbGlzdD0ic2VhcmNoIiwgc3JzZWFyY2g9cXVlcnksIHNybGltaXQ9MSwgZm9ybWF0PSJqc29uIikpCiAgICAgICAgICAgIGhpdHMgPSByLmpzb24oKS5nZXQoInF1ZXJ5Iiwge30pLmdldCgic2VhcmNoIiwgW10pCiAgICAgICAgICAgIGlmIGhpdHM6CiAgICAgICAgICAgICAgICByMiA9IHJlcXVlc3RzLmdldCgiaHR0cHM6Ly9lbi53aWtpcGVkaWEub3JnL3cvYXBpLnBocCIsIGhlYWRlcnM9aCwgdGltZW91dD1zZWxmLnRpbWVvdXQsIHBhcmFtcz1kaWN0KAogICAgICAgICAgICAgICAgICAgIGFjdGlvbj0icXVlcnkiLCBwcm9wPSJleHRyYWN0cyIsIGV4aW50cm89MSwgZXhwbGFpbnRleHQ9MSwgdGl0bGVzPWhpdHNbMF1bInRpdGxlIl0sIGZvcm1hdD0ianNvbiIpKQogICAgICAgICAgICAgICAgcGFnZXMgPSByMi5qc29uKClbInF1ZXJ5Il1bInBhZ2VzIl0KICAgICAgICAgICAgICAgIHR4dCA9IG5leHQoaXRlcihwYWdlcy52YWx1ZXMoKSkpLmdldCgiZXh0cmFjdCIsICIiKQogICAgICAgICAgICAgICAgaWYgbGVuKHR4dCkgPiA0MDoKICAgICAgICAgICAgICAgICAgICBwYXNzYWdlID0gdHh0Wzogc2VsZi5tYXhfY2hhcnNdCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcGFzc2FnZSA9IE5vbmUKICAgICAgICBzZWxmLmNhY2hlW3F1ZXJ5XSA9IHBhc3NhZ2UKICAgICAgICByZXR1cm4gcGFzc2FnZQoKCmNsYXNzIFRocmVlVGllckNvbnRyb2xsZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWwsIHRvaywgZGV2aWNlLCBkZXRlY3RvciwgdGF1X3dhcm5pbmcsIHRhdV9mYXVsdCwgcmV0cmlldmVyPU5vbmUsCiAgICAgICAgICAgICAgICAga19zYW1wbGVzPTQsIHRlbXBlcmF0dXJlPTAuNywgdG9wX3A9MC45LCBtYXhfbmV3X3Rva2Vucz0yNCk6CiAgICAgICAgc2VsZi5tLCBzZWxmLnRvaywgc2VsZi5kZXYsIHNlbGYuZGV0ID0gbW9kZWwsIHRvaywgZGV2aWNlLCBkZXRlY3RvcgogICAgICAgIHNlbGYudHcsIHNlbGYudGYsIHNlbGYucmV0cmlldmVyID0gdGF1X3dhcm5pbmcsIHRhdV9mYXVsdCwgcmV0cmlldmVyCiAgICAgICAgc2VsZi5rLCBzZWxmLlQsIHNlbGYudG9wX3AsIHNlbGYubWF4X25ldyA9IGtfc2FtcGxlcywgdGVtcGVyYXR1cmUsIHRvcF9wLCBtYXhfbmV3X3Rva2VucwoKICAgICMgLS0gZGV0ZWN0b3IgY2FsbCBvbiBhIChwcm9tcHQsIGFuc3dlcikgcGFpcgogICAgZGVmIHJpc2soc2VsZiwgcHJvbXB0LCBwcmVmaXgsIGFuc3dlcik6CiAgICAgICAgciA9IHNjb3JlX2Fuc3dlcihzZWxmLm0sIHNlbGYudG9rLCBzZWxmLmRldiwgcHJvbXB0LCBwcmVmaXggKyBhbnN3ZXIpCiAgICAgICAgaWYgciBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gMS4wLCBOb25lLCBOb25lCiAgICAgICAgRiwgSCA9IHIKICAgICAgICByZXR1cm4gZmxvYXQoc2VsZi5kZXQucHJlZGljdF9wcm9iYShGW05vbmVdLCBIW05vbmVdKVswXSksIEYsIEgKCiAgICBkZWYgcmVzcG9uZChzZWxmLCB1c2VyX3RleHQsIGluaXRpYWw9Tm9uZSk6CiAgICAgICAgIiIiUnVuIHRoZSBjbG9zZWQgbG9vcCBmb3Igb25lIHVzZXIgcmVxdWVzdC4gIGBpbml0aWFsYCA9IG9wdGlvbmFsIChhbnN3ZXIsIHJpc2ssIEYsIEgpIG9mIHRoZSBncmVlZHkgYW5zd2VyCiAgICAgICAgKHJlLWNvbXB1dGVkIGhlcmUgaWYgbm90IHN1cHBsaWVkKS4gIFJldHVybnMgYSBkaWN0IHdpdGggdGhlIGZpbmFsIGFuc3dlciwgdGllciBhbmQgYSBkZWNpc2lvbiB0cmFjZS4iIiIKICAgICAgICBwcm9tcHQsIHByZWZpeCA9IGJ1aWxkX3Byb21wdChzZWxmLnRvaywgdXNlcl90ZXh0KQogICAgICAgIGlmIGluaXRpYWwgaXMgTm9uZToKICAgICAgICAgICAgYW5zMCA9IGdlbmVyYXRlX2Fuc3dlcihzZWxmLm0sIHNlbGYudG9rLCBzZWxmLmRldiwgcHJvbXB0LCBzZWxmLm1heF9uZXcpCiAgICAgICAgICAgIHIwLCBGMCwgSDAgPSBzZWxmLnJpc2socHJvbXB0LCBwcmVmaXgsIGFuczApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYW5zMCwgcjAsIEYwLCBIMCA9IGluaXRpYWwKICAgICAgICB0cmFjZSA9IFtkaWN0KHN0ZXA9ImluaXRpYWxfZ3JlZWR5IiwgYW5zd2VyPWFuczAsIHJpc2s9cjApXQogICAgICAgIGV4cGwgPSBzZWxmLmRldC5leHBsYWluKEYwLCBIMCkgaWYgRjAgaXMgbm90IE5vbmUgZWxzZSB7fQogICAgICAgIGJhc2UgPSBkaWN0KGluaXRpYWxfYW5zd2VyPWFuczAsIGluaXRpYWxfcmlzaz1yMCwgc2Vuc29yX2NvbnRyaWJ1dGlvbnM9ZXhwbCwgdHJhY2U9dHJhY2UpCgogICAgICAgIGlmIHIwIDwgc2VsZi50dzoKICAgICAgICAgICAgcmV0dXJuIGRpY3QoYmFzZSwgdGllcj0iU0FGRSIsIGZpbmFsX2Fuc3dlcj1hbnMwLCBmaW5hbF9yaXNrPXIwKQoKICAgICAgICAjIC0tLS0gVGllciAxICh3YXJuaW5nIHJlZ2ltZSk6IHJpc2stZ3VpZGVkIGNvcnJlY3RpdmUgcmUtZ2VuZXJhdGlvbgogICAgICAgIGlmIHIwIDwgc2VsZi50ZjoKICAgICAgICAgICAgY2FuZHMgPSBbXQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShzZWxmLmspOgogICAgICAgICAgICAgICAgYSA9IGdlbmVyYXRlX2Fuc3dlcihzZWxmLm0sIHNlbGYudG9rLCBzZWxmLmRldiwgcHJvbXB0LCBzZWxmLm1heF9uZXcsIHNhbXBsZT1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZW1wZXJhdHVyZT1zZWxmLlQsIHRvcF9wPXNlbGYudG9wX3ApCiAgICAgICAgICAgICAgICByLCBfLCBfID0gc2VsZi5yaXNrKHByb21wdCwgcHJlZml4LCBhKQogICAgICAgICAgICAgICAgY2FuZHMuYXBwZW5kKChyLCBhKSkKICAgICAgICAgICAgICAgIHRyYWNlLmFwcGVuZChkaWN0KHN0ZXA9InRpZXIxX2NhbmRpZGF0ZSIsIGFuc3dlcj1hLCByaXNrPXIpKQogICAgICAgICAgICByMSwgYTEgPSBtaW4oY2FuZHMsIGtleT1sYW1iZGEgdDogdFswXSkKICAgICAgICAgICAgaWYgcjEgPCBzZWxmLnR3OgogICAgICAgICAgICAgICAgcmV0dXJuIGRpY3QoYmFzZSwgdGllcj0iVElFUjFfUkVTQU1QTEUiLCBmaW5hbF9hbnN3ZXI9YTEsIGZpbmFsX3Jpc2s9cjEpCgogICAgICAgICMgLS0tLSBUaWVyIDIgKGZhdWx0IHJlZ2ltZSwgb3IgVGllciAxIGZhaWxlZCk6IHJldHJpZXZhbC1ncm91bmRlZCByZS1nZW5lcmF0aW9uCiAgICAgICAgaWYgc2VsZi5yZXRyaWV2ZXIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHBhc3NhZ2UgPSBzZWxmLnJldHJpZXZlcih1c2VyX3RleHQpCiAgICAgICAgICAgIGlmIHBhc3NhZ2U6CiAgICAgICAgICAgICAgICBwMiwgXyA9IGJ1aWxkX3Byb21wdChzZWxmLnRvaywgZiJVc2UgdGhpcyBldmlkZW5jZSBpZiByZWxldmFudDoge3Bhc3NhZ2V9XG57dXNlcl90ZXh0fSIpCiAgICAgICAgICAgICAgICBhMiA9IGdlbmVyYXRlX2Fuc3dlcihzZWxmLm0sIHNlbGYudG9rLCBzZWxmLmRldiwgcDIsIHNlbGYubWF4X25ldykKICAgICAgICAgICAgICAgIHIyLCBfLCBfID0gc2VsZi5yaXNrKHAyLCBwcmVmaXgsIGEyKQogICAgICAgICAgICAgICAgdHJhY2UuYXBwZW5kKGRpY3Qoc3RlcD0idGllcjJfcmV0cmlldmFsIiwgYW5zd2VyPWEyLCByaXNrPXIyLCBldmlkZW5jZT1wYXNzYWdlWzoyMDBdKSkKICAgICAgICAgICAgICAgIGlmIHIyIDwgc2VsZi50ZjoKICAgICAgICAgICAgICAgICAgICByZXR1cm4gZGljdChiYXNlLCB0aWVyPSJUSUVSMl9SRVRSSUVWQUwiLCBmaW5hbF9hbnN3ZXI9YTIsIGZpbmFsX3Jpc2s9cjIpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICB0cmFjZS5hcHBlbmQoZGljdChzdGVwPSJ0aWVyMl9yZXRyaWV2YWwiLCBhbnN3ZXI9Tm9uZSwgcmlzaz1Ob25lLCBldmlkZW5jZT0ibm90aGluZyByZXRyaWV2ZWQiKSkKCiAgICAgICAgIyAtLS0tIFRpZXIgMzogY2FsaWJyYXRlZCBhYnN0ZW50aW9uCiAgICAgICAgcmV0dXJuIGRpY3QoYmFzZSwgdGllcj0iVElFUjNfQUJTVEFJTiIsIGZpbmFsX2Fuc3dlcj1BQlNUQUlOX1RFWFQsIGZpbmFsX3Jpc2s9cjApCgo=",
    "tg_eval.py":       "IiIiCkV2YWx1YXRpb24gdXRpbGl0aWVzIGZvciBUUlVFR1VBUkQgdjI6IGxlYWthZ2UtZnJlZSBncm91cGVkIENWLCBjcm9zcy1kYXRhc2V0IHRyYW5zZmVyLCBib290c3RyYXAgQ0lzLAp0aHJlc2hvbGRzLCBzZWxlY3RpdmUtcHJlZGljdGlvbiAocmlzay1jb3ZlcmFnZSkgYW5hbHlzaXMsIGNhbGlicmF0aW9uLCBwbG90cy4KQWxsIGZ1bmN0aW9ucyBvcGVyYXRlIG9uIGZlYXR1cmVzIGNhY2hlZCBieSBleHBlcmltZW50cy9leHRyYWN0X2ZlYXR1cmVzLnB5LgoiIiIKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCAocm9jX2F1Y19zY29yZSwgcm9jX2N1cnZlLCBjb25mdXNpb25fbWF0cml4LCBhY2N1cmFjeV9zY29yZSwgcHJlY2lzaW9uX3Njb3JlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlY2FsbF9zY29yZSwgZjFfc2NvcmUsIGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlKQpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBHcm91cEtGb2xkCgpmcm9tIC50Z19waXBlbGluZSBpbXBvcnQgVHJ1ZUd1YXJkRGV0ZWN0b3IsIFNFTlNPUlMsIEZFQVRVUkVfTkFNRVMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBiYXNpYyBoZWxwZXJzCmRlZiBhdWNfc2FmZSh5LCBzKToKICAgIHkgPSBucC5hc2FycmF5KHkpCiAgICByZXR1cm4gZmxvYXQocm9jX2F1Y19zY29yZSh5LCBzKSkgaWYgbGVuKG5wLnVuaXF1ZSh5KSkgPT0gMiBlbHNlIGZsb2F0KCJuYW4iKQoKCmRlZiBib290c3RyYXBfYXVjKHksIHMsIGdyb3Vwcywgbl9ib290PTUwMCwgc2VlZD0wKToKICAgICIiIkdyb3VwLWxldmVsIGJvb3RzdHJhcCAocmVzYW1wbGVzIHdob2xlIHF1ZXN0aW9ucykgLT4gKGxvLCBoaSkgOTUlIENJLiIiIgogICAgeSwgcywgZ3JvdXBzID0gbnAuYXNhcnJheSh5KSwgbnAuYXNhcnJheShzKSwgbnAuYXNhcnJheShncm91cHMpCiAgICBybmcgPSBucC5yYW5kb20uUmFuZG9tU3RhdGUoc2VlZCkKICAgIHVnID0gbnAudW5pcXVlKGdyb3VwcykKICAgIGlkeCA9IHtnOiBucC53aGVyZShncm91cHMgPT0gZylbMF0gZm9yIGcgaW4gdWd9CiAgICB2YWxzID0gW10KICAgIGZvciBfIGluIHJhbmdlKG5fYm9vdCk6CiAgICAgICAgcGljayA9IG5wLmNvbmNhdGVuYXRlKFtpZHhbZ10gZm9yIGcgaW4gcm5nLmNob2ljZSh1ZywgbGVuKHVnKSldKQogICAgICAgIGEgPSBhdWNfc2FmZSh5W3BpY2tdLCBzW3BpY2tdKQogICAgICAgIGlmIG5vdCBucC5pc25hbihhKToKICAgICAgICAgICAgdmFscy5hcHBlbmQoYSkKICAgIHJldHVybiAoZmxvYXQobnAucGVyY2VudGlsZSh2YWxzLCAyLjUpKSwgZmxvYXQobnAucGVyY2VudGlsZSh2YWxzLCA5Ny41KSkpIGlmIHZhbHMgZWxzZSAobnAubmFuLCBucC5uYW4pCgoKZGVmIGJvb3RzdHJhcF9kZWx0YSh5LCBzX2EsIHNfYiwgZ3JvdXBzLCBuX2Jvb3Q9NTAwLCBzZWVkPTApOgogICAgIiIiUGFpcmVkIGdyb3VwIGJvb3RzdHJhcCBvZiBBVVJPQyhhKSAtIEFVUk9DKGIpLiAgUmV0dXJucyAobWVhbl9kZWx0YSwgbG8sIGhpLCBwX2Ffbm90X2JldHRlcikuIiIiCiAgICB5LCBzX2EsIHNfYiwgZ3JvdXBzID0gbWFwKG5wLmFzYXJyYXksICh5LCBzX2EsIHNfYiwgZ3JvdXBzKSkKICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZShzZWVkKQogICAgdWcgPSBucC51bmlxdWUoZ3JvdXBzKQogICAgaWR4ID0ge2c6IG5wLndoZXJlKGdyb3VwcyA9PSBnKVswXSBmb3IgZyBpbiB1Z30KICAgIGQgPSBbXQogICAgZm9yIF8gaW4gcmFuZ2Uobl9ib290KToKICAgICAgICBwaWNrID0gbnAuY29uY2F0ZW5hdGUoW2lkeFtnXSBmb3IgZyBpbiBybmcuY2hvaWNlKHVnLCBsZW4odWcpKV0pCiAgICAgICAgaWYgbGVuKG5wLnVuaXF1ZSh5W3BpY2tdKSkgPCAyOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGQuYXBwZW5kKHJvY19hdWNfc2NvcmUoeVtwaWNrXSwgc19hW3BpY2tdKSAtIHJvY19hdWNfc2NvcmUoeVtwaWNrXSwgc19iW3BpY2tdKSkKICAgIGQgPSBucC5hcnJheShkKQogICAgcmV0dXJuIGZsb2F0KGQubWVhbigpKSwgZmxvYXQobnAucGVyY2VudGlsZShkLCAyLjUpKSwgZmxvYXQobnAucGVyY2VudGlsZShkLCA5Ny41KSksIGZsb2F0KChkIDw9IDApLm1lYW4oKSkKCgpkZWYgeW91ZGVuX3RocmVzaG9sZCh5LCBzKToKICAgIGZwciwgdHByLCB0aHIgPSByb2NfY3VydmUoeSwgcykKICAgIHJldHVybiBmbG9hdCh0aHJbbnAuYXJnbWF4KHRwciAtIGZwcildKQoKCmRlZiBmcHJfdGhyZXNob2xkKHksIHMsIG1heF9mcHI9MC4xMCk6CiAgICAiIiJTbWFsbGVzdCB0aHJlc2hvbGQgd2hvc2UgZmFsc2UtcG9zaXRpdmUgcmF0ZSBvbiBgeWAgaXMgPD0gbWF4X2ZwciAoaGlnaC1jb25maWRlbmNlICdmYXVsdCcgb3BlcmF0aW5nIHBvaW50KS4iIiIKICAgIGZwciwgdHByLCB0aHIgPSByb2NfY3VydmUoeSwgcykKICAgIG9rID0gbnAud2hlcmUoZnByIDw9IG1heF9mcHIpWzBdCiAgICByZXR1cm4gZmxvYXQodGhyW29rWy0xXV0pIGlmIGxlbihvaykgZWxzZSBmbG9hdCh0aHJbMV0pCgoKZGVmIGNhbGlicmF0ZV90aHJlc2hvbGRzKHksIHJpc2ssIGZhdWx0X21heF9mcHI9MC4xMCk6CiAgICAiIiJXYXJuaW5nIHRocmVzaG9sZCA9IFlvdWRlbi1vcHRpbWFsIHBvaW50OyBGYXVsdCB0aHJlc2hvbGQgPSBoaWdoLXByZWNpc2lvbiBwb2ludCAoRlBSIDw9IGZhdWx0X21heF9mcHIpLgogICAgR3VhcmFudGVlZCB0YXVfZmF1bHQgPj0gdGF1X3dhcm5pbmcuIiIiCiAgICB0dyA9IHlvdWRlbl90aHJlc2hvbGQoeSwgcmlzaykKICAgIHRmID0gbWF4KGZwcl90aHJlc2hvbGQoeSwgcmlzaywgZmF1bHRfbWF4X2ZwciksIHR3KQogICAgcmV0dXJuIHR3LCB0ZgoKCmRlZiBjbGFzc19tZXRyaWNzKHksIHJpc2ssIHRocik6CiAgICB5ID0gbnAuYXNhcnJheSh5KS5hc3R5cGUoaW50KQogICAgcCA9IChucC5hc2FycmF5KHJpc2spID49IHRocikuYXN0eXBlKGludCkKICAgIHRuLCBmcCwgZm4sIHRwID0gY29uZnVzaW9uX21hdHJpeCh5LCBwLCBsYWJlbHM9WzAsIDFdKS5yYXZlbCgpCiAgICByZXR1cm4gZGljdCh0aHJlc2hvbGQ9ZmxvYXQodGhyKSwgYXVyb2M9YXVjX3NhZmUoeSwgcmlzayksIGF1cHJjPWZsb2F0KGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlKHksIHJpc2spKSwKICAgICAgICAgICAgICAgIGFjY3VyYWN5PWZsb2F0KGFjY3VyYWN5X3Njb3JlKHksIHApKSwgcHJlY2lzaW9uPWZsb2F0KHByZWNpc2lvbl9zY29yZSh5LCBwLCB6ZXJvX2RpdmlzaW9uPTApKSwKICAgICAgICAgICAgICAgIHJlY2FsbD1mbG9hdChyZWNhbGxfc2NvcmUoeSwgcCwgemVyb19kaXZpc2lvbj0wKSksIGYxPWZsb2F0KGYxX3Njb3JlKHksIHAsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICAgICAgICAgdHA9aW50KHRwKSwgZnA9aW50KGZwKSwgdG49aW50KHRuKSwgZm49aW50KGZuKSkKCgpkZWYgZWNlKHksIHAsIGJpbnM9MTApOgogICAgeSwgcCA9IG5wLmFzYXJyYXkoeSksIG5wLmFzYXJyYXkocCkKICAgIGVkZ2VzID0gbnAubGluc3BhY2UoMCwgMSwgYmlucyArIDEpCiAgICBlID0gMC4wCiAgICBmb3IgaSBpbiByYW5nZShiaW5zKToKICAgICAgICBtID0gKHAgPj0gZWRnZXNbaV0pICYgKHAgPCBlZGdlc1tpICsgMV0gaWYgaSA8IGJpbnMgLSAxIGVsc2UgcCA8PSBlZGdlc1tpICsgMV0pCiAgICAgICAgaWYgbS5hbnkoKToKICAgICAgICAgICAgZSArPSBtLm1lYW4oKSAqIGFicyh5W21dLm1lYW4oKSAtIHBbbV0ubWVhbigpKQogICAgcmV0dXJuIGZsb2F0KGUpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gcmF3IGJhc2VsaW5lcyAobm8gdHJhaW5pbmcpCmRlZiByYXdfYmFzZWxpbmVzKEYsIHkpOgogICAgIiIiQVVST0Mgb2YgdW50cmFpbmVkIHNpbmdsZSBzaWduYWxzIC8gY2xhc3NpYyBiYXNlbGluZXMuICBIaWdoZXIgcmF3IHZhbHVlIGFzc3VtZWQgPSByaXNraWVyOyB0aGUgcmV2ZXJzZWQKICAgIEFVUk9DICgxLWF1YykgaXMgcmVwb3J0ZWQgdG9vIHNvIGFuIGludmVydGVkIHNlbnNvciBpcyB2aXNpYmxlIGluc3RlYWQgb2YgaGlkZGVuLiIiIgogICAgaWR4ID0ge246IGkgZm9yIGksIG4gaW4gZW51bWVyYXRlKEZFQVRVUkVfTkFNRVMpfQogICAgb3V0ID0ge30KICAgIGZvciBuYW1lIGluIFsiQUFTX21lYW4iLCAiQUFTX21heCIsICJFRU1fbWVhbiIsICJFRU1fbWF4IiwgIkRTVF9tZWFuIiwgIkRTVF9tYXgiLCAiRU5UX21lYW4iLCAiRU5UX21heCIsCiAgICAgICAgICAgICAgICAgIk5MTF9tZWFuIiwgIkxFTiJdOgogICAgICAgIGEgPSBhdWNfc2FmZSh5LCBGWzosIGlkeFtuYW1lXV0pCiAgICAgICAgb3V0W25hbWVdID0gZGljdChhdXJvYz1hLCBhdXJvY19pZl9pbnZlcnRlZD0xIC0gYSkKICAgIHJldHVybiBvdXQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBncm91cGVkIENWIG9mIHRoZSBmdWxsIGRldGVjdG9yCmRlZiBncm91cGVkX2N2KEYsIEgsIHksIGdyb3VwcywgdXNlPVNFTlNPUlMsIGs9NSwgc2VlZD00MiwgKipkZXRfa3cpOgogICAgIiIiTGVha2FnZS1mcmVlIGdyb3VwZWQgSy1mb2xkLiAgSVNQIHByb2JlcywgcHJvYmUtQyBzZWxlY3Rpb24sIGZ1c2lvbiBhbmQgdGhyZXNob2xkcyBhcmUgYWxsIGZpdCBvbiB0aGUKICAgIHRyYWluaW5nIGZvbGQgb25seS4gIFJldHVybnMgcG9vbGVkIG91dC1vZi1mb2xkIHJpc2ssIHBlci1mb2xkIEFVUk9DIGFuZCBwZXItZm9sZCBZb3VkZW4gdGhyZXNob2xkcy4iIiIKICAgIG9vZiA9IG5wLnplcm9zKGxlbih5KSkKICAgIGZvbGRfYXVjLCBmb2xkX3Rociwgc2Vuc29yX29vZiA9IFtdLCBbXSwge3M6IG5wLnplcm9zKGxlbih5KSkgZm9yIHMgaW4gdXNlfQogICAgZ2tmID0gR3JvdXBLRm9sZChuX3NwbGl0cz1taW4oaywgbGVuKG5wLnVuaXF1ZShncm91cHMpKSkpCiAgICBmb3IgdHIsIHRlIGluIGdrZi5zcGxpdChGLCB5LCBncm91cHMpOgogICAgICAgIGRldCA9IFRydWVHdWFyZERldGVjdG9yKHVzZT11c2UsIHNlZWQ9c2VlZCwgKipkZXRfa3cpLmZpdChGW3RyXSwgSFt0cl0sIHlbdHJdLCBncm91cHNbdHJdKQogICAgICAgIG9vZlt0ZV0gPSBkZXQucHJlZGljdF9wcm9iYShGW3RlXSwgSFt0ZV0pCiAgICAgICAgZm9sZF9hdWMuYXBwZW5kKGF1Y19zYWZlKHlbdGVdLCBvb2ZbdGVdKSkKICAgICAgICBmb2xkX3Roci5hcHBlbmQoeW91ZGVuX3RocmVzaG9sZCh5W3RyXSwgZGV0LnRyYWluX3Jpc2tfKSkKICAgICAgICBmb3IgcywgdiBpbiBkZXQuc2Vuc29yX3Njb3JlcyhGW3RlXSwgSFt0ZV0pLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIHMgaW4gc2Vuc29yX29vZjoKICAgICAgICAgICAgICAgIHNlbnNvcl9vb2Zbc11bdGVdID0gdgogICAgcmV0dXJuIGRpY3Qob29mPW9vZiwgZm9sZF9hdWM9Zm9sZF9hdWMsIGZvbGRfdGhyPWZvbGRfdGhyLCBzZW5zb3Jfb29mPXNlbnNvcl9vb2YpCgoKZGVmIHBvb2xlZF9tZXRyaWNzKHksIG9vZiwgZm9sZF90aHIpOgogICAgcmV0dXJuIGNsYXNzX21ldHJpY3MoeSwgb29mLCBmbG9hdChucC5tZWFuKGZvbGRfdGhyKSkpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29tcGxlbWVudGFyaXR5CmRlZiBjb21wbGVtZW50YXJpdHkoeSwgc2Vuc29yX3Njb3JlcywgZnByPTAuMTApOgogICAgIiIiRm9yIGVhY2ggc2Vuc29yOiBmcmFjdGlvbiBvZiBoYWxsdWNpbmF0ZWQgc2FtcGxlcyBpdCBmbGFncyBhdCBhIHRocmVzaG9sZCBnaXZpbmcgYGZwcmAgb24gdGhlIGdyb3VuZGVkCiAgICBzYW1wbGVzLiAgUmVwb3J0cyBob3cgbWFueSBwb3NpdGl2ZXMgYXJlIGNhdWdodCBieSBleGFjdGx5IG9uZSBzZW5zb3IgKHVuaXF1ZSkgYW5kIGJ5IHRoZSB1bmlvbi4iIiIKICAgIHkgPSBucC5hc2FycmF5KHkpCiAgICBmbGFncyA9IHt9CiAgICBmb3IgcywgdiBpbiBzZW5zb3Jfc2NvcmVzLml0ZW1zKCk6CiAgICAgICAgdGhyID0gbnAucGVyY2VudGlsZSh2W3kgPT0gMF0sIDEwMCAqICgxIC0gZnByKSkKICAgICAgICBmbGFnc1tzXSA9ICh2ID4gdGhyKQogICAgcG9zID0geSA9PSAxCiAgICBNID0gbnAuc3RhY2soW2ZsYWdzW3NdW3Bvc10gZm9yIHMgaW4gZmxhZ3NdKQogICAgcmVzID0ge3M6IGRpY3QocmVjYWxsX2F0X2Zwcj1mbG9hdChmbGFnc1tzXVtwb3NdLm1lYW4oKSkpIGZvciBzIGluIGZsYWdzfQogICAgZm9yIGksIHMgaW4gZW51bWVyYXRlKGZsYWdzKToKICAgICAgICBvdGhlcnMgPSBucC5kZWxldGUoTSwgaSwgYXhpcz0wKS5hbnkoMCkgaWYgbGVuKGZsYWdzKSA+IDEgZWxzZSBucC56ZXJvcyhNLnNoYXBlWzFdLCBib29sKQogICAgICAgIHJlc1tzXVsidW5pcXVlX2NhdGNoZXMiXSA9IGZsb2F0KChNW2ldICYgfm90aGVycykubWVhbigpKQogICAgcmVzWyJ1bmlvbl9yZWNhbGwiXSA9IGZsb2F0KE0uYW55KDApLm1lYW4oKSkKICAgIHJlc1siZnByX2xldmVsIl0gPSBmcHIKICAgIHJldHVybiByZXMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBzZWxlY3RpdmUgcHJlZGljdGlvbiAocmlzay1jb3ZlcmFnZSkKZGVmIHJpc2tfY292ZXJhZ2UoY29ycmVjdCwgc2NvcmUsIGNvdmVyYWdlcz0oMS4wLCAwLjksIDAuOCwgMC43LCAwLjYsIDAuNSkpOgogICAgIiIiY29ycmVjdDogMSBpZiB0aGUgYW5zd2VyIGlzIHJpZ2h0LiAgc2NvcmU6IGhpZ2hlciA9IG1vcmUgbGlrZWx5IHdyb25nLiAgS2VlcCB0aGUgKGNvdmVyYWdlKSBmcmFjdGlvbiBvZgogICAgYW5zd2VycyB3aXRoIHRoZSBsb3dlc3Qgc2NvcmU7IHJldHVybiBhY2N1cmFjeSBvbiB0aGUga2VwdCBhbnN3ZXJzLiIiIgogICAgY29ycmVjdCwgc2NvcmUgPSBucC5hc2FycmF5KGNvcnJlY3QpLCBucC5hc2FycmF5KHNjb3JlKQogICAgb3JkZXIgPSBucC5hcmdzb3J0KHNjb3JlLCBraW5kPSJzdGFibGUiKQogICAgcmVzID0ge30KICAgIGZvciBjIGluIGNvdmVyYWdlczoKICAgICAgICBrID0gbWF4KDEsIGludChyb3VuZChjICogbGVuKG9yZGVyKSkpKQogICAgICAgIHJlc1tmIntpbnQoYyoxMDApfSUiXSA9IGZsb2F0KGNvcnJlY3Rbb3JkZXJbOmtdXS5tZWFuKCkpCiAgICAjIGFyZWEgdW5kZXIgcmlzay1jb3ZlcmFnZSAobG93ZXIgaXMgYmV0dGVyKQogICAgY3VtX2VyciA9IG5wLmN1bXN1bSgxIC0gY29ycmVjdFtvcmRlcl0pIC8gbnAuYXJhbmdlKDEsIGxlbihvcmRlcikgKyAxKQogICAgcmVzWyJBVVJDIl0gPSBmbG9hdChjdW1fZXJyLm1lYW4oKSkKICAgIHJldHVybiByZXMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBwbG90dGluZwpkZWYgcGxvdF9yb2MoeSwgY3VydmVzLCBwYXRoLCB0aXRsZSk6CiAgICBpbXBvcnQgbWF0cGxvdGxpYjsgbWF0cGxvdGxpYi51c2UoIkFnZyIpCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CiAgICBwbHQuZmlndXJlKGZpZ3NpemU9KDUuNSwgNSkpCiAgICBmb3IgbmFtZSwgcyBpbiBjdXJ2ZXMuaXRlbXMoKToKICAgICAgICBmcHIsIHRwciwgXyA9IHJvY19jdXJ2ZSh5LCBzKQogICAgICAgIHBsdC5wbG90KGZwciwgdHByLCBsYWJlbD1mIntuYW1lfSAoe2F1Y19zYWZlKHksIHMpOi4zZn0pIiwgbHc9MiBpZiBuYW1lLnN0YXJ0c3dpdGgoIlRSVUVHVUFSRCIpIGVsc2UgMS4yKQogICAgcGx0LnBsb3QoWzAsIDFdLCBbMCwgMV0sICJrLS0iLCBsdz0uOCkKICAgIHBsdC54bGFiZWwoIkZhbHNlIHBvc2l0aXZlIHJhdGUiKTsgcGx0LnlsYWJlbCgiVHJ1ZSBwb3NpdGl2ZSByYXRlIik7IHBsdC50aXRsZSh0aXRsZSkKICAgIHBsdC5sZWdlbmQoZm9udHNpemU9NywgbG9jPSJsb3dlciByaWdodCIpOyBwbHQudGlnaHRfbGF5b3V0KCk7IHBsdC5zYXZlZmlnKHBhdGgsIGRwaT0yMDApOyBwbHQuY2xvc2UoKQoKCmRlZiBwbG90X2JhcnMobGFiZWxzLCB2YWxzLCBlcnJzLCBwYXRoLCB0aXRsZSwgeWxhYmVsPSJBVVJPQyIsIGhsaW5lPTAuNSk6CiAgICBpbXBvcnQgbWF0cGxvdGxpYjsgbWF0cGxvdGxpYi51c2UoIkFnZyIpCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CiAgICBwbHQuZmlndXJlKGZpZ3NpemU9KDcsIDQpKQogICAgcGx0LmJhcihyYW5nZShsZW4odmFscykpLCB2YWxzLCB5ZXJyPWVycnMsIGNhcHNpemU9MywgY29sb3I9IiM0Yzc4YTgiKQogICAgcGx0LmF4aGxpbmUoaGxpbmUsIGNvbG9yPSJrIiwgbHM9Ii0tIiwgbHc9LjgpCiAgICBwbHQueHRpY2tzKHJhbmdlKGxlbih2YWxzKSksIGxhYmVscywgcm90YXRpb249MzUsIGhhPSJyaWdodCIsIGZvbnRzaXplPTgpCiAgICBwbHQueWxhYmVsKHlsYWJlbCk7IHBsdC50aXRsZSh0aXRsZSk7IHBsdC55bGltKDAuMywgMS4wKTsgcGx0LnRpZ2h0X2xheW91dCgpOyBwbHQuc2F2ZWZpZyhwYXRoLCBkcGk9MjAwKTsgcGx0LmNsb3NlKCkKCgpkZWYgcGxvdF9oZWF0bWFwKG1hdCwgcm93cywgY29scywgcGF0aCwgdGl0bGUpOgogICAgaW1wb3J0IG1hdHBsb3RsaWI7IG1hdHBsb3RsaWIudXNlKCJBZ2ciKQogICAgaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAogICAgcGx0LmZpZ3VyZShmaWdzaXplPSg1LjUsIDQuNSkpCiAgICBwbHQuaW1zaG93KG1hdCwgdm1pbj0wLjQsIHZtYXg9MS4wLCBjbWFwPSJ2aXJpZGlzIikKICAgIGZvciBpIGluIHJhbmdlKGxlbihyb3dzKSk6CiAgICAgICAgZm9yIGogaW4gcmFuZ2UobGVuKGNvbHMpKToKICAgICAgICAgICAgaWYgbm90IG5wLmlzbmFuKG1hdFtpLCBqXSk6CiAgICAgICAgICAgICAgICBwbHQudGV4dChqLCBpLCBmInttYXRbaSxqXTouMmZ9IiwgaGE9ImNlbnRlciIsIHZhPSJjZW50ZXIiLCBjb2xvcj0idyIsIGZvbnRzaXplPTkpCiAgICBwbHQueHRpY2tzKHJhbmdlKGxlbihjb2xzKSksIGNvbHMsIHJvdGF0aW9uPTMwLCBoYT0icmlnaHQiKTsgcGx0Lnl0aWNrcyhyYW5nZShsZW4ocm93cykpLCByb3dzKQogICAgcGx0LnhsYWJlbCgidGVzdCBzZXQiKTsgcGx0LnlsYWJlbCgidHJhaW4gc2V0Iik7IHBsdC50aXRsZSh0aXRsZSk7IHBsdC5jb2xvcmJhcihsYWJlbD0iQVVST0MiKQogICAgcGx0LnRpZ2h0X2xheW91dCgpOyBwbHQuc2F2ZWZpZyhwYXRoLCBkcGk9MjAwKTsgcGx0LmNsb3NlKCkKCgpkZWYgcGxvdF9jb25mdXNpb24oY20sIHBhdGgsIHRpdGxlKToKICAgIGltcG9ydCBtYXRwbG90bGliOyBtYXRwbG90bGliLnVzZSgiQWdnIikKICAgIGltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKICAgIHBsdC5maWd1cmUoZmlnc2l6ZT0oMy44LCAzLjQpKQogICAgcGx0Lmltc2hvdyhjbSwgY21hcD0iQmx1ZXMiKQogICAgZm9yIGkgaW4gcmFuZ2UoMik6CiAgICAgICAgZm9yIGogaW4gcmFuZ2UoMik6CiAgICAgICAgICAgIHBsdC50ZXh0KGosIGksIGNtW2ksIGpdLCBoYT0iY2VudGVyIiwgdmE9ImNlbnRlciIsIGZvbnRzaXplPTEyKQogICAgcGx0Lnh0aWNrcyhbMCwgMV0sIFsicHJlZCBncm91bmRlZCIsICJwcmVkIGhhbGx1Yy4iXSwgZm9udHNpemU9OCk7IHBsdC55dGlja3MoWzAsIDFdLCBbImdyb3VuZGVkIiwgImhhbGx1Yy4iXSwgZm9udHNpemU9OCkKICAgIHBsdC50aXRsZSh0aXRsZSwgZm9udHNpemU9OSk7IHBsdC50aWdodF9sYXlvdXQoKTsgcGx0LnNhdmVmaWcocGF0aCwgZHBpPTIwMCk7IHBsdC5jbG9zZSgpCgoKZGVmIHBsb3Rfcmlza19oaXN0KHksIHJpc2ssIHR3LCB0ZiwgcGF0aCwgdGl0bGUpOgogICAgaW1wb3J0IG1hdHBsb3RsaWI7IG1hdHBsb3RsaWIudXNlKCJBZ2ciKQogICAgaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAogICAgcGx0LmZpZ3VyZShmaWdzaXplPSg1LjUsIDMuNSkpCiAgICBwbHQuaGlzdChyaXNrW3kgPT0gMF0sIGJpbnM9MjUsIGFscGhhPS42LCBsYWJlbD0iZ3JvdW5kZWQiLCBjb2xvcj0iIzU5YTE0ZiIpCiAgICBwbHQuaGlzdChyaXNrW3kgPT0gMV0sIGJpbnM9MjUsIGFscGhhPS42LCBsYWJlbD0iaGFsbHVjaW5hdGVkIiwgY29sb3I9IiNlMTU3NTkiKQogICAgaWYgdHcgaXMgbm90IE5vbmU6CiAgICAgICAgcGx0LmF4dmxpbmUodHcsIGNvbG9yPSJvcmFuZ2UiLCBscz0iLS0iLCBsYWJlbD1mIndhcm5pbmcge3R3Oi4yZn0iKQogICAgICAgIHBsdC5heHZsaW5lKHRmLCBjb2xvcj0icmVkIiwgbHM9Ii0tIiwgbGFiZWw9ZiJmYXVsdCB7dGY6LjJmfSIpCiAgICBwbHQueGxhYmVsKCJUUlVFR1VBUkQgcmlzayIpOyBwbHQubGVnZW5kKGZvbnRzaXplPTgpOyBwbHQudGl0bGUodGl0bGUpOyBwbHQudGlnaHRfbGF5b3V0KCkKICAgIHBsdC5zYXZlZmlnKHBhdGgsIGRwaT0yMDApOyBwbHQuY2xvc2UoKQoK",
}

src_hashes = {}
for fname, b64 in _EMBEDDED_SRC.items():
    raw = base64.b64decode(b64)
    with open(os.path.join("src", fname), "wb") as f:
        f.write(raw)
    src_hashes[fname] = hashlib.sha256(raw).hexdigest()[:16]
    print(f"  [OK] Materialized src/{fname:18s} ({len(raw):6,d} bytes | sha256={src_hashes[fname]})")

if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

from src.tg_pipeline import (load_model, build_prompt, score_answer, generate_answer,
                             OnlineSignalMonitor, TrueGuardDetector, SENSORS, FEATURE_NAMES, SENSOR_COLS)
from src.tg_controller import ThreeTierController, WikipediaRetriever, ABSTAIN_TEXT
from src.tg_eval import (auc_safe, bootstrap_auc, bootstrap_delta, calibrate_thresholds,
                         class_metrics, ece, raw_baselines, grouped_cv, pooled_metrics,
                         complementarity, risk_coverage, plot_roc, plot_bars, plot_heatmap,
                         plot_confusion, plot_risk_hist)

AUDIT["embedded_src_sha256"] = src_hashes
AUDIT["stages"]["embed_original_src"] = "COMPLETED"
print("Original TRUEGUARD v2 modules imported cleanly.")


In [ ]:
# ==============================================================================
# DATASET LAYER  --  no Hugging Face dataset *scripts* are used anywhere.
# Every dataset is read from a plain Parquet file (hf_hub_download + pandas), either from the repo's own files or
# from the Hub's auto-converted "refs/convert/parquet" branch.  Label convention (same as the original project):
#     label 0 = grounded / correct / faithful      label 1 = hallucinated / wrong / non-factual
# Every builder returns rows {group, user_text, answer, label} plus a provenance record.
# ==============================================================================
import re, json, hashlib, collections
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files, HfApi

QA_INSTR = "Answer the question briefly in one short sentence.\nQuestion: "          # identical to original tg_data.py
CONVERT = "refs/convert/parquet"

SOURCES = {
    "halueval_dialogue":      dict(repo="pminervini/HaluEval", file="dialogue/data-00000-of-00001.parquet", rev=None),
    "halueval_qa":            dict(repo="pminervini/HaluEval", file="qa/data-00000-of-00001.parquet", rev=None),
    "halueval_summarization": dict(repo="pminervini/HaluEval", file="summarization/data-00000-of-00001.parquet", rev=None),
    "halueval_general":       dict(repo="pminervini/HaluEval", file="general/data-00000-of-00001.parquet", rev=None),
    "truthful_qa":            dict(repo="truthfulqa/truthful_qa", file="generation/validation-00000-of-00001.parquet", rev=None),
    "trivia_qa":              dict(repo="mandarjoshi/trivia_qa", file="rc.nocontext/validation-00000-of-00001.parquet", rev=None),
    "faithdial":              dict(repo="McGill-NLP/FaithDial", file="plain_text/test/0000.parquet", rev=CONVERT),
    "wikibio_gpt3":           dict(repo="potsawee/wiki_bio_gpt3_hallucination", file="data/evaluation-*.parquet", rev=None),
    "fever":                  dict(repo="fever/fever", file="v1.0/paper_dev/0000.parquet", rev=CONVERT),
    "xsum_factuality":        dict(repo="google-research-datasets/xsum_factuality", file="xsum_factuality/train/0000.parquet", rev=CONVERT),
    "xsum_documents":         dict(repo="EdinburghNLP/xsum", file="data/test-00000-of-00001.parquet", rev=None),
}

_API = HfApi()


def _resolve_file(repo, pattern, rev):
    if "*" not in pattern:
        return pattern
    rx = re.compile("^" + re.escape(pattern).replace(r"\*", ".*") + "$")
    hits = [f for f in list_repo_files(repo, repo_type="dataset", revision=rev) if rx.match(f)]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one file matching {pattern} in {repo}, found {hits}")
    return hits[0]


def read_parquet_source(key):
    s = SOURCES[key]
    fname = _resolve_file(s["repo"], s["file"], s["rev"])
    path = hf_hub_download(s["repo"], fname, repo_type="dataset", revision=s["rev"])
    try:
        sha = _API.dataset_info(s["repo"], revision=s["rev"]).sha
    except Exception:
        sha = "unknown"
    return pd.read_parquet(path), dict(repo=s["repo"], file=fname, revision=s["rev"] or "main", commit_sha=sha)


def _need(df, cols, name):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise RuntimeError(f"[{name}] schema mismatch: missing columns {miss}; found {list(df.columns)}")


def _s(x):
    return "" if x is None or (isinstance(x, float) and np.isnan(x)) else str(x).strip()


# ----------------------------------------------------------------------------- paired benchmarks (same as original)
def build_halueval_qa(cap, seed=42):
    df, src = read_parquet_source("halueval_qa"); _need(df, ["question", "right_answer", "hallucinated_answer"], "halueval_qa")
    n_raw = len(df); rng = np.random.RandomState(seed); out = []; excl = 0
    for gi, i in enumerate(rng.permutation(n_raw)[: (cap or n_raw)]):
        r = df.iloc[int(i)]
        a0, a1 = _s(r["right_answer"]), _s(r["hallucinated_answer"])
        if not a0 or not a1:
            excl += 1; continue
        ut = QA_INSTR + _s(r["question"])
        out += [dict(group=gi, user_text=ut, answer=a0, label=0), dict(group=gi, user_text=ut, answer=a1, label=1)]
    return out, dict(src, n_raw=n_raw, unit="question pairs", n_selected_units=len(out) // 2, excluded=excl,
                     note="pair = (right_answer=0, hallucinated_answer=1); group = question")


def build_halueval_dialogue(cap, seed=42):
    df, src = read_parquet_source("halueval_dialogue"); _need(df, ["dialogue_history", "right_response", "hallucinated_response"], "halueval_dialogue")
    n_raw = len(df); rng = np.random.RandomState(seed); out = []; excl = 0
    for gi, i in enumerate(rng.permutation(n_raw)[: (cap or n_raw)]):
        r = df.iloc[int(i)]
        a0, a1 = _s(r["right_response"]), _s(r["hallucinated_response"])
        if not a0 or not a1:
            excl += 1; continue
        ut = "Continue the dialogue with one short reply.\nDialogue:\n" + _s(r["dialogue_history"])[-1200:]
        out += [dict(group=gi, user_text=ut, answer=a0, label=0), dict(group=gi, user_text=ut, answer=a1, label=1)]
    return out, dict(src, n_raw=n_raw, unit="dialogue pairs", n_selected_units=len(out) // 2, excluded=excl,
                     note="pair = (right_response=0, hallucinated_response=1); group = dialogue")


def build_halueval_summarization(cap, seed=42):
    df, src = read_parquet_source("halueval_summarization"); _need(df, ["document", "right_summary", "hallucinated_summary"], "halueval_summarization")
    n_raw = len(df); rng = np.random.RandomState(seed); out = []; excl = 0
    for gi, i in enumerate(rng.permutation(n_raw)[: (cap or n_raw)]):
        r = df.iloc[int(i)]
        a0, a1 = _s(r["right_summary"]), _s(r["hallucinated_summary"])
        if not a0 or not a1:
            excl += 1; continue
        ut = "Summarize the document in one or two sentences.\nDocument: " + _s(r["document"])[:1200]
        out += [dict(group=gi, user_text=ut, answer=a0, label=0), dict(group=gi, user_text=ut, answer=a1, label=1)]
    return out, dict(src, n_raw=n_raw, unit="document pairs", n_selected_units=len(out) // 2, excluded=excl,
                     note="pair = (right_summary=0, hallucinated_summary=1); group = document")


def build_truthful_qa(cap, seed=42):
    df, src = read_parquet_source("truthful_qa"); _need(df, ["question", "best_answer", "incorrect_answers"], "truthful_qa")
    n_raw = len(df); rng = np.random.RandomState(seed); out = []; excl = 0
    for gi, i in enumerate(rng.permutation(n_raw)[: (cap or n_raw)]):
        r = df.iloc[int(i)]
        inc = [_s(x) for x in list(r["incorrect_answers"]) if _s(x)]
        best = _s(r["best_answer"])
        if not inc or not best:
            excl += 1; continue
        wrong = inc[int(rng.randint(len(inc)))]
        if wrong.lower() == best.lower():
            excl += 1; continue
        ut = QA_INSTR + _s(r["question"])
        out += [dict(group=gi, user_text=ut, answer=best, label=0), dict(group=gi, user_text=ut, answer=wrong, label=1)]
    return out, dict(src, n_raw=n_raw, unit="question pairs", n_selected_units=len(out) // 2, excluded=excl,
                     note="pair = (best_answer=0, one randomly drawn incorrect_answer=1); a question is excluded only if either side is empty/identical")


# ----------------------------------------------------------------------------- HaluEval-General (single response, native label)
def build_halueval_general(cap, seed=42):
    df, src = read_parquet_source("halueval_general")
    _need(df, ["user_query", "chatgpt_response", "hallucination"], "halueval_general")
    vals = collections.Counter(_s(v).lower() for v in df["hallucination"])
    if not set(vals) <= {"yes", "no"}:
        raise RuntimeError(f"[halueval_general] unexpected 'hallucination' values: {dict(vals)}")
    idx = np.arange(len(df))
    if cap:
        idx = np.random.RandomState(seed).permutation(len(df))[:cap]
    out = []; excl = 0
    for gi, i in enumerate(idx):
        r = df.iloc[int(i)]
        resp, q = _s(r["chatgpt_response"]), _s(r["user_query"])
        if not resp or not q:
            excl += 1; continue
        out.append(dict(group=int(gi), user_text="Follow the instruction and answer clearly.\nQuery: " + q[:1200], answer=resp,
                        label=1 if _s(r["hallucination"]).lower() == "yes" else 0))
    return out, dict(src, n_raw=len(df), unit="responses", n_selected_units=len(out), excluded=excl, raw_label_values=dict(vals),
                     note="native human label hallucination=yes ->1 / no ->0; group = query; response is truncated to 48 answer tokens by score_answer")


# ----------------------------------------------------------------------------- FaithDial (verified field semantics)
def build_faithdial(cap=None, seed=42):
    df, src = read_parquet_source("faithdial")
    _need(df, ["dialog_idx", "response", "original_response", "history", "knowledge", "BEGIN"], "faithdial")
    out = []; counts = collections.Counter()
    for _, r in df.iterrows():
        tags = set(str(t) for t in list(r["BEGIN"]))
        orig, edited = _s(r["original_response"]), _s(r["response"])
        # BEGIN annotates the ORIGINAL (Wizard-of-Wikipedia) response; an empty original_response means the WoW response
        # was already faithful and `response` is that same (lightly edited) turn.  The human-rewritten `response` of an
        # unfaithful turn is NEVER used as a hallucination example.
        text = orig if orig else edited
        if "Hallucination" in tags:
            lbl, kind = 1, "any_hallucination_tag"
        elif "Entailment" in tags:
            lbl, kind = 0, "entailment_no_hallucination"
        else:
            counts["excluded_generic_only"] += 1; continue
        if not text:
            counts["excluded_empty_text"] += 1; continue
        hist = [_s(h) for h in list(r["history"])]
        ut = ("Knowledge: " + _s(r["knowledge"])[:900] + "\nDialogue:\n" + "\n".join(hist[-4:]) +
              "\nContinue the dialogue with one short reply grounded in the knowledge.")
        out.append(dict(group=int(r["dialog_idx"]), user_text=ut, answer=text, label=lbl,
                        strict=int(("Hallucination" in tags) != ("Entailment" in tags))))
        counts[kind] += 1
    # strict flag = 1 for unambiguous rows (exactly one of Hallucination / Entailment present); used for a secondary metric
    return out, dict(src, n_raw=len(df), unit="dialogue turns", n_selected_units=len(out), excluded=int(sum(v for k, v in counts.items() if k.startswith("excluded"))),
                     label_construction=dict(counts),
                     note="text = original_response (WoW turn) or, if empty, response; label 1 if BEGIN contains Hallucination, 0 if Entailment and no Hallucination; "
                          "Generic-only turns excluded; group = dialog_idx (791 conversations)")


# ----------------------------------------------------------------------------- WikiBio-GPT3 (SelfCheckGPT)
def _wb_subject(wiki_text):
    t = _s(wiki_text)
    m = re.match(r"^(.{3,80}?)\s*\(", t)
    if m:
        return m.group(1).strip()
    return " ".join(t.split()[:4])


def build_wikibio(cap=None, seed=42):
    df, src = read_parquet_source("wikibio_gpt3")
    _need(df, ["gpt3_sentences", "annotation", "wiki_bio_test_idx", "wiki_bio_text"], "wikibio_gpt3")
    out = []; counts = collections.Counter(); excl = 0
    for _, r in df.iterrows():
        sents, ann = [str(x) for x in list(r["gpt3_sentences"])], [str(x) for x in list(r["annotation"])]
        if len(sents) != len(ann):
            raise RuntimeError("[wikibio_gpt3] sentence/annotation length mismatch")
        subj, gid = _wb_subject(r["wiki_bio_text"]), int(r["wiki_bio_test_idx"])
        for k, (snt, a) in enumerate(zip(sents, ann)):
            if a not in ("accurate", "minor_inaccurate", "major_inaccurate"):
                raise RuntimeError(f"[wikibio_gpt3] unexpected annotation {a}")
            if not snt.strip():
                excl += 1; continue
            ctx = " ".join(sents[:k])[-600:]
            ut = f"Write a short factual Wikipedia-style biography of {subj}." + (f"\nSo far: {ctx}\nWrite the next sentence." if ctx else "\nWrite the first sentence.")
            out.append(dict(group=gid, user_text=ut, answer=snt.strip(), label=0 if a == "accurate" else 1, major=int(a == "major_inaccurate"), annotation=a))
            counts[a] += 1
    return out, dict(src, n_raw=int(sum(len(list(x)) for x in df["gpt3_sentences"])), unit="sentences", n_selected_units=len(out), excluded=excl, label_construction=dict(counts),
                     note="sentence-level; accurate->0, minor_inaccurate+major_inaccurate->1 (SelfCheckGPT NonFact); extra metric NonFact* = accurate vs major only; group = article (238)")


# ----------------------------------------------------------------------------- FEVER (claim-verification transfer, de-duplicated)
def build_fever(cap=None, seed=42):
    df, src = read_parquet_source("fever")
    _need(df, ["id", "label", "claim"], "fever")
    n_rows = len(df)
    d = df[df["label"].isin(["SUPPORTS", "REFUTES"])].drop_duplicates(subset="id")     # raw file repeats a claim once per evidence set
    d = d.sort_values("id")
    if cap:
        d = d.iloc[np.random.RandomState(seed).permutation(len(d))[:cap]]
    out = [dict(group=int(r["id"]), user_text="State one verifiable fact.", answer=_s(r["claim"]), label=0 if r["label"] == "SUPPORTS" else 1)
           for _, r in d.iterrows() if _s(r["claim"])]
    return out, dict(src, n_raw=int(n_rows), unit="unique claims", n_selected_units=len(out), excluded=int(n_rows - len(out)),
                     label_counts=dict(collections.Counter(d["label"])),
                     note="SUPPORTS->0 / REFUTES->1; NOT ENOUGH INFO removed; duplicate evidence rows collapsed by claim id; human-written claims, NOT model generations => claim-verification (truthfulness-transfer) benchmark; group = claim id")


# ----------------------------------------------------------------------------- XSum factuality (aggregated human judgments + source article)
def build_xsum(cap=None, seed=42):
    df, src = read_parquet_source("xsum_factuality")
    _need(df, ["bbcid", "system", "summary", "is_factual"], "xsum_factuality")
    docs, dsrc = read_parquet_source("xsum_documents"); _need(docs, ["id", "document"], "xsum_documents")
    doc_by_id = {str(i): d for i, d in zip(docs["id"], docs["document"])}
    votes = collections.defaultdict(list)
    for r in df.itertuples(index=False):
        votes[(int(r.bbcid), str(r.system), str(r.summary))].append(int(r.is_factual))
    out = []; tie = 0; nodoc = 0; vote_hist = collections.Counter(len(v) for v in votes.values())
    for (bid, system, summ), v in votes.items():
        p = float(np.mean(v))
        if p == 0.5:
            tie += 1; continue                           # tie (only possible with an even number of judgments)
        doc = doc_by_id.get(str(bid))
        if doc is None:
            nodoc += 1; continue
        out.append(dict(group=int(bid), user_text="Summarize the document in one or two sentences.\nDocument: " + _s(doc)[:1200], answer=_s(summ),
                        label=0 if p > 0.5 else 1, system=system, n_votes=len(v)))
    if nodoc > 0.02 * len(votes):
        raise RuntimeError(f"[xsum_factuality] source article missing for {nodoc}/{len(votes)} summaries")
    return out, dict(src, n_raw=len(df), n_unique_summaries=len(votes), judgments_per_summary=dict(vote_hist), unit="summaries", n_selected_units=len(out),
                     excluded=tie + nodoc, ties_removed=tie, missing_article=nodoc, documents_source=dsrc,
                     note="majority vote over the independent judgments of each (article, system, summary): mean(is_factual)>0.5 ->0, <0.5 ->1, ties dropped; "
                          "prompt includes the real XSum source article (joined on bbcid); group = article")


BUILDERS = dict(halueval_dialogue=build_halueval_dialogue, halueval_qa=build_halueval_qa, halueval_summarization=build_halueval_summarization,
                truthful_qa=build_truthful_qa, halueval_general=build_halueval_general, faithdial=build_faithdial,
                wikibio_gpt3=build_wikibio, fever=build_fever, xsum_factuality=build_xsum)


# ----------------------------------------------------------------------------- TriviaQA (mitigation) -- same helpers as original tg_data.py
import string as _string


def norm_text(s):
    s = s.lower()
    s = "".join(c for c in s if c not in _string.punctuation)
    return " ".join(s.split())


def load_triviaqa(n, seed=42, offset=0):
    df, src = read_parquet_source("trivia_qa")
    idx = np.random.RandomState(seed).permutation(len(df))[offset:offset + n]
    items = []
    for i in idx:
        r = df.iloc[int(i)]
        ans = r["answer"]
        aliases = [a for a in list(ans["normalized_aliases"]) if a]
        if aliases:
            items.append(dict(id=int(i), question=_s(r["question"]), user_text=QA_INSTR + _s(r["question"]), gold=aliases))
    return items


def is_correct(answer, gold):
    a = norm_text(answer)
    return int(any(re.search(r"\b" + re.escape(g) + r"\b", a) for g in gold))


def rows_sha256(rows):
    h = hashlib.sha256()
    for r in rows:
        h.update(json.dumps([r["group"], r["user_text"], r["answer"], r["label"]], sort_keys=True).encode())
    return h.hexdigest()[:16]


In [ ]:
# STEP 6: Download, Build, and Validate All 10 Datasets (Manifest + Schema + Balance Checks)
from tabulate import tabulate

DATASETS = {}
MANIFEST = {}

print("=== LOADING AND VALIDATING ALL 9 DETECTION / TRANSFER DATASETS ===")
t0_all = time.time()
for name in IN_DOMAIN_DATASETS + UNSEEN_ZEROSHOT_DATASETS:
    role = "IN_DOMAIN_CV_AND_POOL" if name in IN_DOMAIN_DATASETS else (
        "UNSEEN_CLAIM_VERIFICATION_TRANSFER" if name == "fever" else "UNSEEN_ZEROSHOT_TEST"
    )
    t0 = time.time()
    rows, meta = BUILDERS[name](DATASET_CAPS[name], seed=SEED)
    labels = [r["label"] for r in rows]
    c0, c1 = labels.count(0), labels.count(1)
    n_groups = len(set(r["group"] for r in rows))
    assert len(rows) > 0, f"{name} produced 0 rows!"
    assert c0 > 0 and c1 > 0, f"{name} is single-class (c0={c0}, c1={c1})!"
    h = rows_sha256(rows)
    meta.update(role=role, n_rows=len(rows), n_groups=n_groups, label_0_grounded=c0, label_1_hallucinated=c1,
                prevalence=round(c1 / len(rows), 4), rows_sha256=h, load_sec=round(time.time() - t0, 2))
    DATASETS[name] = rows
    MANIFEST[name] = meta
    print(f"  [OK] {name:24s} | {role:34s} | rows={len(rows):5d} | groups={n_groups:5d} | 0:{c0:4d} 1:{c1:4d} | sha={h}")

print("\n=== LOADING AND VALIDATING TRIVIAQA (MITIGATION) ===")
t0 = time.time()
TRIVIA_CAL  = load_triviaqa(TRIVIAQA_N_CAL,  seed=SEED, offset=0)
TRIVIA_TEST = load_triviaqa(TRIVIAQA_N_TEST, seed=SEED, offset=TRIVIAQA_N_CAL)
assert len(set(x["id"] for x in TRIVIA_CAL) & set(x["id"] for x in TRIVIA_TEST)) == 0, "TriviaQA cal/test overlap!"
MANIFEST["trivia_qa"] = dict(role="MITIGATION_CLOSED_BOOK", n_cal=len(TRIVIA_CAL), n_test=len(TRIVIA_TEST),
                             disjoint_verified=True, load_sec=round(time.time() - t0, 2))
print(f"  [OK] trivia_qa               | MITIGATION_CLOSED_BOOK             | cal={len(TRIVIA_CAL)} test={len(TRIVIA_TEST)} (disjoint=True)")

manifest_path = os.path.join(DIRS["results"], "dataset_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2)

tbl = [[k, v["role"], v.get("n_raw", "—"), v.get("n_rows", f"{v.get('n_cal')}+{v.get('n_test')}"),
        v.get("n_groups", "—"), v.get("label_0_grounded", "—"), v.get("label_1_hallucinated", "—"), v.get("excluded", 0)]
       for k, v in MANIFEST.items()]
print("\n" + tabulate(tbl, headers=["Dataset", "Role", "Raw", "Usable Rows", "Groups", "Label 0", "Label 1", "Excluded"], tablefmt="grid"))
AUDIT["stages"]["load_and_validate_10_datasets"] = f"COMPLETED ({len(MANIFEST)}/10 datasets in {time.time()-t0_all:.1f}s)"


In [ ]:
# STEP 7: Load Frozen TinyLlama-1.1B with Eager Attention & Verify Genuine 4-Sensor Tensors
t0 = time.time()
model, tok, DEVICE = load_model(MODEL_NAME, device="auto")
for p in model.parameters():
    p.requires_grad = False

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
attn_impl        = getattr(model.config, "_attn_implementation", getattr(model.config, "attn_implementation", "unknown"))

print(f"Model:            {MODEL_NAME}")
print(f"Device:           {DEVICE} | dtype: {next(model.parameters()).dtype}")
print(f"Parameters:       {total_params:,} total | {trainable_params} trainable (100% FROZEN)")
print(f"Attention impl:   {attn_impl}")
assert trainable_params == 0, "Foundation model must remain 100% frozen!"
assert attn_impl == "eager", f"Expected eager attention for AAS sensor, got {attn_impl}"

# Genuine 1-sample tensor validation check
_sample = DATASETS["halueval_qa"][0]
_p, _pre = build_prompt(tok, _sample["user_text"])
_res = score_answer(model, tok, DEVICE, _p, _pre + _sample["answer"])
assert _res is not None, "score_answer returned None on sanity sample!"
_F, _H = _res
assert _F.shape == (10,), f"Expected F shape (10,), got {_F.shape}"
assert _H.shape == (4, model.config.hidden_size), f"Expected H shape (4, {model.config.hidden_size}), got {_H.shape}"
assert np.all(np.isfinite(_F)) and np.all(np.isfinite(_H)), "Non-finite sensor values!"
print(f"Sanity forward pass OK: F.shape={_F.shape}, H.shape={_H.shape}, AAS={_F[0]:.4f}, EEM={_F[2]:.4f}, DST={_F[4]:.4f}, NLL={_F[8]:.4f}")
AUDIT["stages"]["load_frozen_tinyllama"] = f"COMPLETED ({DEVICE}, eager, 0 trainable, {time.time()-t0:.1f}s)"


In [ ]:
# STEP 8: Genuine GPU 4-Sensor Feature Extraction with Verified Incremental Checkpointing
# Checkpoints store (model_name, pipeline_version, rows_sha256) so stale or mismatched caches are NEVER reused.
import gc

def extract_or_resume(name, rows, batch_size=CHECKPOINT_BATCH):
    expected_meta = {
        "dataset": name,
        "model": MODEL_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "rows_sha256": MANIFEST[name]["rows_sha256"],
        "n_input_rows": len(rows),
    }
    final_npz   = os.path.join(DIRS["checkpoints"], f"{name}.npz")
    final_meta  = os.path.join(DIRS["checkpoints"], f"{name}.meta.json")
    partial_npz = os.path.join(DIRS["checkpoints"], f"{name}.partial.npz")
    partial_meta = os.path.join(DIRS["checkpoints"], f"{name}.partial.meta.json")

    # 1. Check completed verified checkpoint
    if os.path.exists(final_npz) and os.path.exists(final_meta):
        saved_meta = json.load(open(final_meta, "r", encoding="utf-8"))
        if all(saved_meta.get(k) == v for k, v in expected_meta.items()):
            d = np.load(final_npz)
            F, H, y, g = d["F"].astype(np.float64), d["H"].astype(np.float32), d["y"].astype(int), d["groups"].astype(np.int64)
            extra = {k: d[k] for k in d.files if k not in ("F", "H", "y", "groups")}
            assert F.shape[1] == 10 and H.shape[1:] == (4, model.config.hidden_size)
            print(f"  [VERIFIED CACHE] {name:24s} | rows={len(y):5d} | H={H.shape} | sha={saved_meta['rows_sha256']}")
            return F, H, y, g, extra, "CACHED_VERIFIED"
        else:
            print(f"  [CACHE MISMATCH] {name} metadata differs from current run — re-extracting fresh features.")

    # 2. Check partial resume checkpoint
    start_idx = 0
    Fs, Hs, ys, gs = [], [], [], []
    extra_keys = [k for k in ("strict", "major") if k in rows[0]]
    extras_acc = {k: [] for k in extra_keys}
    skipped_empty = 0

    if os.path.exists(partial_npz) and os.path.exists(partial_meta):
        pm = json.load(open(partial_meta, "r", encoding="utf-8"))
        if all(pm.get(k) == v for k, v in expected_meta.items()):
            d = np.load(partial_npz)
            Fs = list(d["F"]); Hs = list(d["H"]); ys = list(d["y"]); gs = list(d["groups"])
            for k in extra_keys:
                if k in d.files:
                    extras_acc[k] = list(d[k])
            start_idx = int(pm["processed_input_rows"])
            skipped_empty = int(pm.get("skipped_empty", 0))
            print(f"  [RESUMING PARTIAL] {name}: resuming from input row {start_idx}/{len(rows)} ({len(ys)} extracted)")

    t0 = time.perf_counter()
    print(f"  [EXTRACTING ON {DEVICE}] {name}: {len(rows) - start_idx} remaining rows...")
    for idx in range(start_idx, len(rows)):
        r = rows[idx]
        p, pre = build_prompt(tok, r["user_text"])
        res = score_answer(model, tok, DEVICE, p, pre + r["answer"])
        if res is None:
            skipped_empty += 1
        else:
            F_i, H_i = res
            if np.all(np.isfinite(F_i)) and np.all(np.isfinite(H_i)):
                Fs.append(F_i.astype(np.float32))
                Hs.append(H_i.astype(np.float16))
                ys.append(int(r["label"]))
                gs.append(int(r["group"]))
                for k in extra_keys:
                    extras_acc[k].append(int(r[k]))
            else:
                skipped_empty += 1

        done = idx + 1
        if (done - start_idx) % batch_size == 0 or done == len(rows):
            dt = max(time.perf_counter() - t0, 1e-3)
            rate_min = (done - start_idx) / dt * 60.0
            eta_min = (len(rows) - done) / max((done - start_idx) / dt, 1e-3) / 60.0
            print(f"    [{name}] {done:5d}/{len(rows):5d} | valid={len(ys):5d} | {rate_min:5.1f} seq/min | ETA {eta_min:4.1f} min", flush=True)
            save_dict = dict(F=np.asarray(Fs, np.float32), H=np.asarray(Hs, np.float16),
                             y=np.asarray(ys, np.int64), groups=np.asarray(gs, np.int64))
            for k in extra_keys:
                save_dict[k] = np.asarray(extras_acc[k], np.int64)
            np.savez_compressed(partial_npz, **save_dict)
            with open(partial_meta, "w", encoding="utf-8") as mf:
                json.dump(dict(expected_meta, processed_input_rows=done, valid_rows=len(ys), skipped_empty=skipped_empty), mf, indent=2)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Finalize atomic checkpoint
    save_dict = dict(F=np.asarray(Fs, np.float32), H=np.asarray(Hs, np.float16),
                     y=np.asarray(ys, np.int64), groups=np.asarray(gs, np.int64))
    for k in extra_keys:
        save_dict[k] = np.asarray(extras_acc[k], np.int64)
    np.savez_compressed(final_npz, **save_dict)
    total_sec = time.perf_counter() - t0
    with open(final_meta, "w", encoding="utf-8") as mf:
        json.dump(dict(expected_meta, valid_rows=len(ys), skipped_empty=skipped_empty,
                       extraction_sec=round(total_sec, 2), throughput_seq_per_min=round((len(rows) - start_idx) / max(total_sec, 1e-3) * 60.0, 2)), mf, indent=2)
    for tmp in (partial_npz, partial_meta):
        if os.path.exists(tmp):
            os.remove(tmp)
    gc.collect()
    extra = {k: save_dict[k] for k in extra_keys}
    return save_dict["F"].astype(np.float64), save_dict["H"].astype(np.float32), save_dict["y"].astype(int), save_dict["groups"].astype(np.int64), extra, f"COMPLETED ({total_sec/60:.1f} min)"

FEATURES = {}
EXTRAS   = {}
EXTRACT_STATUS = {}
t_ext_all = time.time()
for name in IN_DOMAIN_DATASETS + UNSEEN_ZEROSHOT_DATASETS:
    F, H, y, g, ex, st = extract_or_resume(name, DATASETS[name])
    # For paired benchmarks, keep only complete pairs (both label=0 and label=1 present), matching original evaluate_detection.py
    if name in ("halueval_dialogue", "halueval_qa", "halueval_summarization", "truthful_qa"):
        ug, cnt = np.unique(g, return_counts=True)
        keep = np.isin(g, ug[cnt == 2])
        F, H, y, g = F[keep], H[keep], y[keep], g[keep]
    FEATURES[name] = (F, H, y, g)
    EXTRAS[name]   = ex
    EXTRACT_STATUS[name] = dict(status=st, n_rows=int(len(y)), n_groups=int(len(np.unique(g))), F_shape=list(F.shape), H_shape=list(H.shape))

AUDIT["extraction"] = EXTRACT_STATUS
AUDIT["stages"]["feature_extraction_9_datasets"] = f"COMPLETED ({len(FEATURES)}/9 in {(time.time()-t_ext_all)/60:.1f} min)"
print(f"\nAll 9 detection/generalization datasets extracted & verified in {(time.time()-t_ext_all)/60:.1f} min.")


In [ ]:
# ==============================================================================
# EVALUATION LAYER -- thin orchestration around the ORIGINAL src.tg_pipeline / src.tg_eval code.
# Only one engineering addition: the ISP probe stage (the expensive part of TrueGuardDetector.fit) is computed ONCE per
# training set and shared by all sensor subsets.  detector_from_stage() reproduces TrueGuardDetector.fit() line by line,
# and cell "equivalence test" proves identical outputs against the untouched original class.
# ==============================================================================
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.model_selection import GroupKFold

from src.tg_pipeline import TrueGuardDetector, SENSORS, FEATURE_NAMES, SENSOR_COLS
from src.tg_eval import (auc_safe, bootstrap_auc, bootstrap_delta, calibrate_thresholds, class_metrics, ece, raw_baselines,
                         youden_threshold, complementarity, risk_coverage, plot_roc, plot_bars, plot_heatmap, plot_confusion,
                         plot_risk_hist)

FIDX = {n: i for i, n in enumerate(FEATURE_NAMES)}
SUBSETS = {
    "full": SENSORS,
    "ISP_only": ("ISP",), "AAS_only": ("AAS",), "EEM_only": ("EEM",), "DST_only": ("DST",),
    "without_ISP": ("AAS", "EEM", "DST"), "without_AAS": ("ISP", "EEM", "DST"),
    "without_EEM": ("ISP", "AAS", "DST"), "without_DST": ("ISP", "AAS", "EEM"),
}


def isp_stage(base, H, y, groups):
    """ISP part of TrueGuardDetector.fit (C chosen by grouped inner CV on the training rows only; final probes on all training rows)."""
    from sklearn.metrics import roc_auc_score as _auc
    H = np.asarray(H, np.float32); y = np.asarray(y).astype(int)
    splits = base._splits(y, groups)
    best = (-1, None, None)
    for C in base.C_grid:
        oof = base._oof_probe(H, y, splits, C)
        auc = _auc(y, oof)
        if auc > best[0]:
            best = (auc, C, oof)
    probes = [base._lr(best[1]).fit(H[:, k], y) for k in range(H.shape[1])]
    return dict(auc=best[0], C=best[1], isp_train=best[2], probes=probes)


def detector_from_stage(use, stage, F, y, **kw):
    """Same statements as TrueGuardDetector.fit after the ISP block, re-using a shared ISP stage."""
    det = TrueGuardDetector(use=use, **kw)
    F = np.asarray(F, np.float64); y = np.asarray(y).astype(int)
    isp_train = None
    if "ISP" in det.use:
        det.probe_auc_inner_, det.probe_C_, isp_train = stage["auc"], stage["C"], stage["isp_train"]
        det.probes_ = stage["probes"]
    Z = det._design(F, isp_train)
    det.fusion_ = det._lr(det.fusion_C).fit(Z, y)
    det.train_risk_ = det.fusion_.predict_proba(Z)[:, 1]
    det.y_train_ = y
    return det


def fit_all_subsets(F, H, y, groups, subsets=SUBSETS, **kw):
    base = TrueGuardDetector(use=SENSORS, **kw)
    stage = isp_stage(base, H, y, groups) if any("ISP" in u for u in subsets.values()) else None
    return {name: detector_from_stage(use, stage, F, y, **kw) for name, use in subsets.items()}, stage


def grouped_cv_all(F, H, y, groups, subsets=SUBSETS, k=5, seed=42, log=None):
    """Original grouped_cv() (GroupKFold; probes, probe-C, fusion and fold thresholds fit on the training fold only) evaluated for every
    sensor subset on identical folds, sharing the per-fold ISP stage."""
    n = len(y)
    out = {s: dict(oof=np.zeros(n), fold_auc=[], fold_thr=[], sensor_oof={x: np.zeros(n) for x in u}) for s, u in subsets.items()}
    gkf = GroupKFold(n_splits=min(k, len(np.unique(groups))))
    folds = list(gkf.split(F, y, groups))
    for fi, (tr, te) in enumerate(folds):
        if log: log(f"      fold {fi+1}/{len(folds)}: train={len(tr)} test={len(te)} (groups disjoint: {len(set(groups[tr]) & set(groups[te])) == 0})")
        assert len(set(groups[tr]) & set(groups[te])) == 0, "group leakage between train and test fold"
        dets, _ = fit_all_subsets(F[tr], H[tr], y[tr], groups[tr], subsets, seed=seed)
        for s, det in dets.items():
            out[s]["oof"][te] = det.predict_proba(F[te], H[te])
            out[s]["fold_auc"].append(auc_safe(y[te], out[s]["oof"][te]))
            out[s]["fold_thr"].append(youden_threshold(y[tr], det.train_risk_))
            for x, v in det.sensor_scores(F[te], H[te]).items():
                if x in out[s]["sensor_oof"]:
                    out[s]["sensor_oof"][x][te] = v
    return out


def brier(y, p):
    return float(brier_score_loss(np.asarray(y).astype(int), np.asarray(p)))


def evaluate_in_domain(name, F, H, y, g, fig_dir, n_boot=500, log=print):
    """Mirror of experiments/evaluate_detection.py for ONE dataset (all numbers from out-of-fold predictions)."""
    cv = grouped_cv_all(F, H, y, g, log=log)
    full = cv["full"]; oof = full["oof"]
    auc_ci = bootstrap_auc(y, oof, g, n_boot=n_boot)
    thr = float(np.mean(full["fold_thr"]))
    pm = class_metrics(y, oof, thr)
    pm.update(auroc_fold_mean=float(np.mean(full["fold_auc"])), auroc_fold_std=float(np.std(full["fold_auc"])),
              auroc_ci95=list(auc_ci), ece=ece(y, oof), brier=brier(y, oof), prevalence=float(np.mean(y)))
    single = {s: dict(auroc=auc_safe(y, cv[f"{s}_only"]["oof"]), auroc_ci95=list(bootstrap_auc(y, cv[f"{s}_only"]["oof"], g, n_boot=n_boot)),
                      fold_mean=float(np.mean(cv[f"{s}_only"]["fold_auc"])), fold_std=float(np.std(cv[f"{s}_only"]["fold_auc"]))) for s in SENSORS}
    loso = {}
    for s in SENSORS:
        r = cv[f"without_{s}"]
        d_mean, d_lo, d_hi, p = bootstrap_delta(y, oof, r["oof"], g, n_boot=n_boot)
        loso[f"without_{s}"] = dict(auroc=auc_safe(y, r["oof"]), delta_vs_full=-d_mean, full_minus_ablated_ci95=[d_lo, d_hi], p_sensor_not_helping=p)
    best_s = max(single, key=lambda k: single[k]["auroc"])
    d_mean, d_lo, d_hi, p_val = bootstrap_delta(y, oof, cv[f"{best_s}_only"]["oof"], g, n_boot=n_boot)
    d2 = bootstrap_delta(y, oof, cv["ISP_only"]["oof"], g, n_boot=n_boot)
    comp = complementarity(y, full["sensor_oof"])
    S = np.column_stack([full["sensor_oof"][s] for s in SENSORS])
    corr = {SENSORS[i]: {SENSORS[j]: float(np.corrcoef(S[:, i], S[:, j])[0, 1]) for j in range(4)} for i in range(4)}
    raw = raw_baselines(F, y)
    diag = []
    for kname, v in raw.items():
        if v["auroc"] < 0.48:
            diag.append(f"{kname}: raw AUROC={v['auroc']:.3f} (<0.5) -> inverted on {name} (inverted AUROC={v['auroc_if_inverted']:.3f})")
    if d_mean <= 0:
        diag.append(f"Fusion did NOT beat best single sensor ({best_s}={single[best_s]['auroc']:.3f} vs fusion={pm['auroc']:.3f})")
    elif p_val > 0.05:
        diag.append(f"Fusion gain over {best_s} is {d_mean:+.4f} but 95% CI [{d_lo:.4f},{d_hi:.4f}] crosses 0 (p={p_val:.2f})")
    # figures
    curves = {"TRUEGUARD (4-sensor)": oof, **{f"{s} only": cv[f"{s}_only"]["oof"] for s in SENSORS},
              "Baseline: NLL": F[:, FIDX["NLL_mean"]], "Baseline: Entropy": F[:, FIDX["ENT_mean"]]}
    plot_roc(y, curves, f"{fig_dir}/roc_{name}.png", f"ROC - {name}")
    plot_confusion(np.array([[pm["tn"], pm["fp"]], [pm["fn"], pm["tp"]]]), f"{fig_dir}/cm_{name}.png", f"Confusion - {name}")
    tw, tf = calibrate_thresholds(y, oof)
    plot_risk_hist(y, oof, tw, tf, f"{fig_dir}/risk_{name}.png", f"Out-of-fold risk - {name}")
    labels = ["Full (4)"] + [f"-{s}" for s in SENSORS] + [f"{s} only" for s in SENSORS]
    vals = [pm["auroc"]] + [loso[f"without_{s}"]["auroc"] for s in SENSORS] + [single[s]["auroc"] for s in SENSORS]
    plot_bars(labels, vals, [0] * len(vals), f"{fig_dir}/ablation_{name}.png", f"Ablation - {name}")
    result = dict(n_rows=int(len(y)), n_groups=int(len(np.unique(g))), fusion=pm, single_sensor_trained=single, ablation_leave_one_out=loso,
                  fusion_vs_best_single=dict(best_single=best_s, gain=d_mean, gain_ci95=[d_lo, d_hi], p_value=p_val),
                  fusion_vs_ISP_only=dict(gain=d2[0], gain_ci95=[d2[1], d2[2]], p_value=d2[3]),
                  fusion_without_ISP_auroc=auc_safe(y, cv["without_ISP"]["oof"]),
                  raw_untrained_baselines=raw, complementarity_at_10pct_fpr=comp, sensor_correlation=corr, diagnostics=diag,
                  per_fold_auroc=[float(x) for x in full["fold_auc"]])
    arrays = dict(y=y, groups=g, oof_full=oof, oof_isp=cv["ISP_only"]["oof"], oof_aas=cv["AAS_only"]["oof"],
                  oof_eem=cv["EEM_only"]["oof"], oof_dst=cv["DST_only"]["oof"])
    return result, arrays


def fit_pool(parts, log=print):
    """parts: dict name -> (F,H,y,g).  Group ids are made globally unique before pooling."""
    Fs, Hs, ys, gs, off = [], [], [], [], 0
    for k, (name, (F, H, y, g)) in enumerate(parts.items()):
        Fs.append(F); Hs.append(H); ys.append(y); gs.append(g.astype(np.int64) + (k + 1) * 10 ** 9); off += len(y)
    F, H, y, g = np.vstack(Fs), np.concatenate(Hs), np.concatenate(ys), np.concatenate(gs)
    log(f"   pooled training set: {len(y)} rows, {len(np.unique(g))} groups, prevalence={y.mean():.3f}, H={H.shape}")
    sub = {k: SUBSETS[k] for k in ("full", "ISP_only", "AAS_only", "EEM_only", "DST_only", "without_ISP")}
    dets, stage = fit_all_subsets(F, H, y, g, sub)
    tw, tf = calibrate_thresholds(y, dets["full"].train_risk_)
    return dets, dict(tau_warning=float(tw), tau_fault=float(tf), n_train=int(len(y)), probe_C=float(stage["C"]), probe_inner_auc=float(stage["auc"]),
                      pool_sizes={k: int(len(v[2])) for k, v in parts.items()})


def evaluate_unseen(name, F, H, y, g, dets, thr, fig_dir, extra=None, n_boot=500):
    """Zero-shot evaluation: nothing from this dataset has touched a scaler, probe, fusion layer or threshold."""
    risk = dets["full"].predict_proba(F, H)
    tw, tf = thr["tau_warning"], thr["tau_fault"]
    res = dict(n_rows=int(len(y)), n_groups=int(len(np.unique(g))), prevalence=float(np.mean(y)),
               fusion=dict(auroc=auc_safe(y, risk), auroc_ci95=list(bootstrap_auc(y, risk, g, n_boot=n_boot)), auprc=float(average_precision_score(y, risk)),
                           ece=ece(y, risk), brier=brier(y, risk), at_tau_warning=class_metrics(y, risk, tw), at_tau_fault=class_metrics(y, risk, tf)))
    scores = {k: dets[k].predict_proba(F, H) for k in ("ISP_only", "AAS_only", "EEM_only", "DST_only", "without_ISP")}
    res["variants"] = {k: dict(auroc=auc_safe(y, v), auroc_ci95=list(bootstrap_auc(y, v, g, n_boot=n_boot)), auprc=float(average_precision_score(y, v)))
                       for k, v in scores.items()}
    d = bootstrap_delta(y, risk, scores["ISP_only"], g, n_boot=n_boot)
    res["fusion_vs_ISP_only"] = dict(gain=d[0], gain_ci95=[d[1], d[2]], p_value=d[3])
    res["raw_untrained_baselines"] = raw_baselines(F, y)
    if extra:
        for mname, mask in extra.items():
            mask = np.asarray(mask, bool)
            if mask.sum() > 10 and len(np.unique(y[mask])) == 2:
                res[mname] = dict(n=int(mask.sum()), auroc=auc_safe(y[mask], risk[mask]), isp_only_auroc=auc_safe(y[mask], scores["ISP_only"][mask]))
    curves = {"TRUEGUARD (4-sensor)": risk, "ISP only": scores["ISP_only"], "Baseline: NLL": F[:, FIDX["NLL_mean"]], "Baseline: Entropy": F[:, FIDX["ENT_mean"]]}
    plot_roc(y, curves, f"{fig_dir}/zeroshot_roc_{name}.png", f"Zero-shot ROC - {name}")
    m = res["fusion"]["at_tau_warning"]
    plot_confusion(np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]]), f"{fig_dir}/zeroshot_cm_{name}.png", f"Zero-shot confusion @tau_warn - {name}")
    plot_risk_hist(y, risk, tw, tf, f"{fig_dir}/zeroshot_risk_{name}.png", f"Zero-shot risk - {name}")
    return res, dict(y=y, groups=g, risk_full=risk, risk_isp=scores["ISP_only"], risk_scalar=scores["without_ISP"])


def reliability_table(y, p, bins=10):
    y, p = np.asarray(y), np.asarray(p); e = np.linspace(0, 1, bins + 1); rows = []
    for i in range(bins):
        m = (p >= e[i]) & ((p < e[i + 1]) if i < bins - 1 else (p <= e[i + 1]))
        if m.any():
            rows.append(dict(bin=f"{e[i]:.1f}-{e[i+1]:.1f}", n=int(m.sum()), mean_pred=float(p[m].mean()), frac_pos=float(y[m].mean())))
    return rows


In [ ]:
# STEP 10: In-Domain 5-Fold Group-Preserving Cross-Validation, Sensor Ablations & Calibration
# Strictly evaluated on the 5 designated in-domain datasets; zero leakage across groups or inner probe-to-fusion stacking.
import joblib

IN_DOMAIN_RESULTS = {}
IN_DOMAIN_PREDS   = {}
t0_id = time.time()

for name in IN_DOMAIN_DATASETS:
    F, H, y, g = FEATURES[name]
    print(f"\n=== IN-DOMAIN 5-FOLD GROUP CV: {name} (rows={len(y)}, groups={len(np.unique(g))}, prev={y.mean():.3f}) ===")
    res, arr = evaluate_in_domain(name, F, H, y, g, DIRS["figures"], n_boot=N_BOOT, log=print)
    res["reliability_bins"] = reliability_table(y, arr["oof_full"])
    IN_DOMAIN_RESULTS[name] = res
    IN_DOMAIN_PREDS[name]   = arr
    np.savez_compressed(os.path.join(DIRS["predictions"], f"in_domain_oof_{name}.npz"), **arr)
    f = res["fusion"]; s = res["single_sensor_trained"]
    print(f"  => Fusion AUROC={f['auroc']:.4f} [{f['auroc_ci95'][0]:.4f},{f['auroc_ci95'][1]:.4f}] | "
          f"ISP={s['ISP']['auroc']:.4f} AAS={s['AAS']['auroc']:.4f} EEM={s['EEM']['auroc']:.4f} DST={s['DST']['auroc']:.4f} | "
          f"AUPRC={f['auprc']:.4f} F1={f['f1']:.4f} ECE={f['ece']:.4f} Brier={f['brier']:.4f}")

# Pairwise cross-dataset transfer matrix among the 5 in-domain datasets (train on A -> test on B)
print("\n=== IN-DOMAIN 5x5 CROSS-DATASET TRANSFER MATRIX ===")
in_names = list(IN_DOMAIN_DATASETS)
mat = np.full((len(in_names), len(in_names)), np.nan)
IN_DOMAIN_TRANSFER = {}
for i, tr_name in enumerate(in_names):
    Ftr, Htr, ytr, gtr = FEATURES[tr_name]
    det = TrueGuardDetector(seed=SEED).fit(Ftr, Htr, ytr, gtr)
    tw, tf = calibrate_thresholds(ytr, det.train_risk_)
    IN_DOMAIN_TRANSFER[tr_name] = dict(tau_warning=float(tw), tau_fault=float(tf), transfers={})
    for j, te_name in enumerate(in_names):
        if i == j:
            mat[i, j] = IN_DOMAIN_RESULTS[tr_name]["fusion"]["auroc"]
            continue
        Fte, Hte, yte, gte = FEATURES[te_name]
        r = det.predict_proba(Fte, Hte)
        m = class_metrics(yte, r, tw)
        m["auroc_ci95"] = list(bootstrap_auc(yte, r, gte, n_boot=N_BOOT))
        IN_DOMAIN_TRANSFER[tr_name]["transfers"][te_name] = m
        mat[i, j] = m["auroc"]
plot_heatmap(mat, in_names, in_names, os.path.join(DIRS["figures"], "in_domain_transfer_matrix.png"),
             "In-Domain Cross-Dataset AUROC (diagonal = 5-fold Group CV)")

AUDIT["stages"]["in_domain_5fold_cv_and_ablations"] = f"COMPLETED (5/5 datasets in {(time.time()-t0_id)/60:.1f} min)"


In [ ]:
# STEP 11: Fit Pooled In-Domain Detector & Evaluate Strictly Unseen Zero-Shot Benchmarks
# GUARANTEE: FaithDial, WikiBio-GPT3, FEVER, and XSum Factuality contribute ZERO rows to training, scaling,
#            hyperparameter selection, or threshold calibration.
t0_zs = time.time()
print("=== FITTING POOLED IN-DOMAIN DETECTOR (5 IN-DOMAIN DATASETS ONLY) ===")
in_pool = {k: FEATURES[k] for k in IN_DOMAIN_DATASETS}
POOLED_DETS, POOLED_META = fit_pool(in_pool, log=print)
print(f"Pooled detector ready: probe_C={POOLED_META['probe_C']}, inner_AUC={POOLED_META['probe_inner_auc']:.4f}, "
      f"tau_warning={POOLED_META['tau_warning']:.4f}, tau_fault={POOLED_META['tau_fault']:.4f}")

# Save trained pooled classifiers for reproducibility
joblib.dump({"detectors": POOLED_DETS, "meta": POOLED_META}, os.path.join(DIRS["models"], "trueguard_pooled_in_domain.joblib"))

UNSEEN_RESULTS = {}
print("\n=== STRICTLY UNSEEN ZERO-SHOT EVALUATION (4 HELD-OUT DATASETS) ===")
for name in UNSEEN_ZEROSHOT_DATASETS:
    F, H, y, g = FEATURES[name]
    extra_masks = {}
    if name == "faithdial" and "strict" in EXTRAS[name]:
        extra_masks["unambiguous_strict_subset"] = EXTRAS[name]["strict"] == 1
    if name == "wikibio_gpt3" and "major" in EXTRAS[name]:
        extra_masks["accurate_vs_major_only_NonFactStar"] = (y == 0) | (EXTRAS[name]["major"] == 1)

    res, arr = evaluate_unseen(name, F, H, y, g, POOLED_DETS, POOLED_META, DIRS["figures"], extra=extra_masks, n_boot=N_BOOT)
    res["benchmark_type"] = "CLAIM_VERIFICATION_TRANSFER" if name == "fever" else "HALLUCINATION_ZEROSHOT_TRANSFER"
    res["reliability_bins"] = reliability_table(y, arr["risk_full"])
    UNSEEN_RESULTS[name] = res
    np.savez_compressed(os.path.join(DIRS["predictions"], f"unseen_zeroshot_{name}.npz"), **arr)

    f = res["fusion"]; v = res["variants"]; m = f["at_tau_warning"]
    tag = "[CLAIM-VERIF TRANSFER]" if name == "fever" else "[UNSEEN ZERO-SHOT]    "
    print(f"  {tag} {name:18s} | n={len(y):5d} | Fusion AUROC={f['auroc']:.4f} [{f['auroc_ci95'][0]:.4f},{f['auroc_ci95'][1]:.4f}] | "
          f"ISP={v['ISP_only']['auroc']:.4f} | AUPRC={f['auprc']:.4f} | F1@tw={m['f1']:.4f} | ECE={f['ece']:.4f}")

AUDIT["stages"]["unseen_zeroshot_4_datasets"] = f"COMPLETED (4/4 datasets in {(time.time()-t0_zs)/60:.1f} min)"


In [ ]:
# STEP 12: TriviaQA Closed-Book Generation & Three-Tier Risk-Based Mitigation
# Uses the exact original evaluate_mitigation.py protocol:
#   - Calibration split (n_cal=250): greedy-generate, extract real 4 sensors, label vs TriviaQA gold aliases,
#     fit TrueGuardDetector and calibrate (tau_warning, tau_fault) strictly on calibration split.
#   - Test split (n_test=250, disjoint): run ThreeTierController (SAFE / TIER1_RESAMPLE / TIER2_RETRIEVAL / TIER3_ABSTAIN)
#     and compute before/after hallucination rates, answer coverage, tier transitions, and risk-coverage curves.
#   - Also evaluates the pooled in-domain detector's zero-shot transfer on TriviaQA generations for comparison.

def _gen_and_score_tqa(items, cache_tag, max_new=24):
    ckpt = os.path.join(DIRS["checkpoints"], f"triviaqa_{cache_tag}_{len(items)}.npz")
    meta_p = os.path.join(DIRS["checkpoints"], f"triviaqa_{cache_tag}_{len(items)}.meta.json")
    exp = dict(model=MODEL_NAME, pipeline_version=PIPELINE_VERSION, tag=cache_tag, n=len(items), seed=SEED)
    if os.path.exists(ckpt) and os.path.exists(meta_p):
        sm = json.load(open(meta_p, "r", encoding="utf-8"))
        if all(sm.get(k) == v for k, v in exp.items()):
            d = np.load(ckpt, allow_pickle=True)
            print(f"  [VERIFIED CACHE] triviaqa_{cache_tag}: {len(d['y'])} generated & scored answers")
            return d["F"].astype(np.float64), d["H"].astype(np.float32), d["y"].astype(int), list(d["ans"]), list(d["kept_idx"].astype(int))
    Fs, Hs, ys, ans_list, kept_idx = [], [], [], [], []
    t0 = time.perf_counter()
    for i, it in enumerate(items):
        p, pre = build_prompt(tok, it["user_text"])
        a = generate_answer(model, tok, DEVICE, p, max_new_tokens=max_new)
        r = score_answer(model, tok, DEVICE, p, pre + a)
        if r is None:
            continue
        F_i, H_i = r
        Fs.append(F_i.astype(np.float32)); Hs.append(H_i.astype(np.float16))
        ys.append(1 - is_correct(a, it["gold"]))   # 1 = hallucinated / wrong
        ans_list.append(a); kept_idx.append(i)
        if (i + 1) % 50 == 0 or (i + 1) == len(items):
            print(f"    [triviaqa_{cache_tag}] {i+1}/{len(items)} generated & scored ({time.perf_counter()-t0:.1f}s)", flush=True)
    F_arr, H_arr, y_arr = np.asarray(Fs, np.float32), np.asarray(Hs, np.float16), np.asarray(ys, np.int64)
    np.savez_compressed(ckpt, F=F_arr, H=H_arr, y=y_arr, ans=np.asarray(ans_list, dtype=object), kept_idx=np.asarray(kept_idx, np.int64))
    json.dump(dict(exp, valid=len(ys)), open(meta_p, "w", encoding="utf-8"), indent=2)
    return F_arr.astype(np.float64), H_arr.astype(np.float32), y_arr.astype(int), ans_list, kept_idx

t0_mit = time.time()
print(f"=== TRIVIAQA CLOSED-BOOK MITIGATION (n_cal={len(TRIVIA_CAL)}, n_test={len(TRIVIA_TEST)}) ===")
F_cal, H_cal, y_cal, ans_cal, _       = _gen_and_score_tqa(TRIVIA_CAL,  "cal")
F_te,  H_te,  y_te,  ans_te, kept_te  = _gen_and_score_tqa(TRIVIA_TEST, "test")
qs_te = [TRIVIA_TEST[i] for i in kept_te]

assert len(np.unique(y_cal)) == 2 and min(y_cal.sum(), len(y_cal) - y_cal.sum()) >= 5, "Calibration split lacks both classes!"
det_tqa = TrueGuardDetector(seed=SEED).fit(F_cal, H_cal, y_cal)
tw_tqa, tf_tqa = calibrate_thresholds(y_cal, det_tqa.train_risk_)
joblib.dump({"detector": det_tqa, "tau_warning": tw_tqa, "tau_fault": tf_tqa}, os.path.join(DIRS["models"], "trueguard_triviaqa_mitigation.joblib"))
print(f"  Calibrated thresholds on calibration split ONLY: tau_warning={tw_tqa:.4f}, tau_fault={tf_tqa:.4f}")

risk_te = det_tqa.predict_proba(F_te, H_te)
risk_te_pooled_transfer = POOLED_DETS["full"].predict_proba(F_te, H_te)
print(f"  Test detection AUROC on TinyLlama's own generations: calibrated={auc_safe(y_te, risk_te):.4f} | pooled_in_domain_transfer={auc_safe(y_te, risk_te_pooled_transfer):.4f}")

retr = WikipediaRetriever() if USE_WIKIPEDIA_RETRIEVAL else None
ctrl = ThreeTierController(model, tok, DEVICE, det_tqa, tw_tqa, tf_tqa, retriever=retr)

tiers, final_correct, answered, records = [], [], [], []
t1_fixed = t2_fixed = 0
for i, it in enumerate(qs_te):
    res = ctrl.respond(it["user_text"], initial=(ans_te[i], float(risk_te[i]), F_te[i], H_te[i]))
    tier = res["tier"]; tiers.append(tier)
    if tier == "TIER3_ABSTAIN":
        answered.append(0); final_correct.append(0)
    else:
        answered.append(1)
        c = is_correct(res["final_answer"], it["gold"])
        final_correct.append(c)
        if y_te[i] == 1 and c == 1:
            if tier == "TIER1_RESAMPLE": t1_fixed += 1
            elif tier == "TIER2_RETRIEVAL": t2_fixed += 1
    if len(records) < 30:
        records.append(dict(question=it["question"], gold=it["gold"][:3], greedy_answer=ans_te[i],
                            greedy_correct=int(1 - y_te[i]), initial_risk=float(risk_te[i]),
                            tier=tier, final_answer=res["final_answer"],
                            final_correct=None if tier == "TIER3_ABSTAIN" else int(final_correct[-1]),
                            sensor_contributions=res["sensor_contributions"]))
    if (i + 1) % 50 == 0 or (i + 1) == len(qs_te):
        print(f"    [ThreeTierController] {i+1}/{len(qs_te)} mitigated...", flush=True)

answered = np.asarray(answered, bool); final_correct = np.asarray(final_correct, int)
greedy_correct = 1 - y_te
before_halluc = float(y_te.mean())
after_halluc_answered = float(1 - final_correct[answered].mean()) if answered.any() else 0.0
after_halluc_system   = float(((1 - final_correct) * answered).mean())
cov = float(answered.mean())

sel = dict(
    TRUEGUARD_calibrated=risk_coverage(greedy_correct, risk_te),
    TRUEGUARD_pooled_transfer=risk_coverage(greedy_correct, risk_te_pooled_transfer),
    NLL_baseline=risk_coverage(greedy_correct, F_te[:, FIDX["NLL_mean"]]),
    Entropy_baseline=risk_coverage(greedy_correct, F_te[:, FIDX["ENT_mean"]]),
)
MITIGATION_REPORT = dict(
    model=MODEL_NAME, n_cal=int(len(y_cal)), n_test=int(len(y_te)),
    tau_warning=float(tw_tqa), tau_fault=float(tf_tqa),
    test_detection_auroc_on_own_generations=auc_safe(y_te, risk_te),
    test_detection_auroc_pooled_transfer=auc_safe(y_te, risk_te_pooled_transfer),
    before=dict(accuracy=float(greedy_correct.mean()), hallucination_rate=before_halluc),
    after=dict(coverage=cov, accuracy_among_answered=1.0 - after_halluc_answered,
               hallucination_rate_among_answered=after_halluc_answered,
               hallucination_rate_system_wide=after_halluc_system,
               relative_hallucination_reduction_among_answered=(before_halluc - after_halluc_answered) / max(before_halluc, 1e-9),
               relative_hallucination_reduction_system_wide=(before_halluc - after_halluc_system) / max(before_halluc, 1e-9)),
    tier_counts={t: int((np.asarray(tiers) == t).sum()) for t in ("SAFE", "TIER1_RESAMPLE", "TIER2_RETRIEVAL", "TIER3_ABSTAIN")},
    greedy_hallucinations_corrected=dict(by_tier1_resample=int(t1_fixed), by_tier2_retrieval=int(t2_fixed)),
    selective_prediction_accuracy_at_coverage=sel,
    examples=records,
)
with open(os.path.join(DIRS["results"], "mitigation_report.json"), "w", encoding="utf-8") as f:
    json.dump(MITIGATION_REPORT, f, indent=2)
np.savez_compressed(os.path.join(DIRS["predictions"], "triviaqa_test_predictions.npz"),
                    y=y_te, risk_calibrated=risk_te, risk_pooled=risk_te_pooled_transfer,
                    answered=answered.astype(int), final_correct=final_correct)

# Risk-coverage figure
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
covs = [100, 90, 80, 70, 60, 50]
plt.figure(figsize=(6, 4))
for lbl, key in [("TRUEGUARD (calibrated)", "TRUEGUARD_calibrated"), ("TRUEGUARD (pooled transfer)", "TRUEGUARD_pooled_transfer"),
                 ("Baseline: NLL", "NLL_baseline"), ("Baseline: Entropy", "Entropy_baseline")]:
    plt.plot(covs, [sel[key][f"{c}%"] for c in covs], marker="o", label=f"{lbl} (AURC={sel[key]['AURC']:.3f})")
plt.gca().invert_xaxis(); plt.xlabel("Answer Coverage (%)"); plt.ylabel("Selective Accuracy")
plt.title("TriviaQA Risk-Coverage Curve"); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(DIRS["figures"], "triviaqa_risk_coverage.png"), dpi=200); plt.close()

print(f"  Before mitigation: hallucination={before_halluc*100:.1f}% (acc={(1-before_halluc)*100:.1f}%)")
print(f"  After mitigation:  coverage={cov*100:.1f}% | halluc_among_answered={after_halluc_answered*100:.1f}% | system_halluc={after_halluc_system*100:.1f}%")
print(f"  Tier counts:       {MITIGATION_REPORT['tier_counts']} | Fixed: T1={t1_fixed}, T2={t2_fixed}")
AUDIT["stages"]["triviaqa_mitigation"] = f"COMPLETED ({len(y_cal)} cal + {len(y_te)} test in {(time.time()-t0_mit)/60:.1f} min)"


In [ ]:
# STEP 13: Genuine Per-Token Inference Latency Profiling (Matches Original profile_real_latency.py)
def _sync():
    if str(DEVICE).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()

t0_lat = time.time()
lat_prompts = [build_prompt(tok, it["user_text"])[0] for it in load_triviaqa(LATENCY_N_PROMPTS, seed=7)]
det_lat = POOLED_DETS["full"]  # Real trained detector (not a dummy)

# Warmup
_ids0 = tok(lat_prompts[0], return_tensors="pt").input_ids.to(model.device)
with torch.no_grad():
    _ = model(_ids0, output_hidden_states=True, output_attentions=True)
_sync()

@torch.no_grad()
def _time_vanilla():
    t0 = time.perf_counter(); toks = 0
    for p in lat_prompts:
        ids = tok(p, return_tensors="pt").input_ids.to(model.device)
        past = None; cur = ids
        for _ in range(LATENCY_N_TOKENS):
            out = model(cur, past_key_values=past, use_cache=True, output_hidden_states=False, output_attentions=False)
            past = out.past_key_values; cur = out.logits[:, -1:, :].argmax(-1); toks += 1
    _sync()
    return (time.perf_counter() - t0) * 1000 / toks

@torch.no_grad()
def _time_trueguard_online():
    t0 = time.perf_counter(); t_math = 0.0; toks = 0
    mon = OnlineSignalMonitor()
    for p in lat_prompts:
        mon.reset()
        ids = tok(p, return_tensors="pt").input_ids.to(model.device)
        past = None; cur = ids
        for _ in range(LATENCY_N_TOKENS):
            out = model(cur, past_key_values=past, use_cache=True, output_hidden_states=True, output_attentions=True)
            past = out.past_key_values; nxt = out.logits[:, -1:, :].argmax(-1)
            _sync(); tm0 = time.perf_counter()
            mon.update(out, int(nxt[0, 0]))
            F_i, H_i = mon.features()
            _ = det_lat.predict_proba(F_i[None], H_i[None])
            _sync(); t_math += time.perf_counter() - tm0
            cur = nxt; toks += 1
    _sync()
    return (time.perf_counter() - t0) * 1000 / toks, t_math * 1000 / toks

@torch.no_grad()
def _time_postpass():
    t0 = time.perf_counter(); toks = 0
    for p in lat_prompts:
        ids = tok(p, return_tensors="pt").input_ids.to(model.device)
        gen = model.generate(ids, max_new_tokens=LATENCY_N_TOKENS, min_new_tokens=LATENCY_N_TOKENS,
                             do_sample=False, pad_token_id=tok.pad_token_id)
        ans = tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)
        r = score_answer(model, tok, DEVICE, p, " " + ans)
        if r is not None:
            _ = det_lat.predict_proba(r[0][None], r[1][None])
        toks += LATENCY_N_TOKENS
    _sync()
    return (time.perf_counter() - t0) * 1000 / toks

@torch.no_grad()
def _time_5x():
    t0 = time.perf_counter(); toks = 0
    for p in lat_prompts[:max(5, LATENCY_N_PROMPTS // 3)]:
        ids = tok(p, return_tensors="pt").input_ids.to(model.device)
        for _ in range(5):
            _ = model.generate(ids, max_new_tokens=LATENCY_N_TOKENS, min_new_tokens=LATENCY_N_TOKENS,
                               do_sample=True, temperature=0.7, pad_token_id=tok.pad_token_id)
        toks += LATENCY_N_TOKENS
    _sync()
    return (time.perf_counter() - t0) * 1000 / toks

v_ms = _time_vanilla()
tg_ms, math_ms = _time_trueguard_online()
post_ms = _time_postpass()
five_ms = _time_5x()

LATENCY_REPORT = dict(
    model=MODEL_NAME, device=str(DEVICE), n_prompts=LATENCY_N_PROMPTS, tokens_per_prompt=LATENCY_N_TOKENS,
    vanilla_ms_per_token=v_ms, trueguard_online_ms_per_token=tg_ms, trueguard_sensor_math_only_ms_per_token=math_ms,
    trueguard_online_overhead_ms_per_token=tg_ms - v_ms, trueguard_online_relative_overhead=(tg_ms - v_ms) / max(v_ms, 1e-9),
    trueguard_postpass_ms_per_token=post_ms, trueguard_postpass_overhead_ms_per_token=post_ms - v_ms,
    multi_sample_5x_ms_per_token=five_ms, speedup_online_vs_5x=five_ms / max(tg_ms, 1e-9),
    speedup_postpass_vs_5x=five_ms / max(post_ms, 1e-9),
)
with open(os.path.join(DIRS["results"], "real_latency.json"), "w", encoding="utf-8") as f:
    json.dump(LATENCY_REPORT, f, indent=2)

print("=== GENUINE INFERENCE LATENCY PROFILE ===")
print(f"  Vanilla decode:         {v_ms:6.2f} ms/token")
print(f"  TRUEGUARD online:       {tg_ms:6.2f} ms/token (overhead={LATENCY_REPORT['trueguard_online_relative_overhead']*100:.1f}%, sensor math={math_ms:.2f} ms)")
print(f"  TRUEGUARD post-pass:    {post_ms:6.2f} ms/token (overhead={(post_ms-v_ms)/max(v_ms,1e-9)*100:.1f}%)")
print(f"  5x multi-sample:        {five_ms:6.2f} ms/token")
print(f"  Speedup vs 5x sampling: online={LATENCY_REPORT['speedup_online_vs_5x']:.2f}x | post-pass={LATENCY_REPORT['speedup_postpass_vs_5x']:.2f}x")
AUDIT["stages"]["latency_profiling"] = f"COMPLETED ({time.time()-t0_lat:.1f}s)"


In [ ]:
# STEP 14: Save Master JSON Report, Print Comparison Tables & Strict Execution Audit
AUDIT["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

FINAL_REPORT = {
    "audit": AUDIT,
    "manifest": MANIFEST,
    "in_domain_5fold_cv": IN_DOMAIN_RESULTS,
    "in_domain_transfer_5x5": IN_DOMAIN_TRANSFER,
    "pooled_in_domain_meta": POOLED_META,
    "unseen_zeroshot_4_datasets": UNSEEN_RESULTS,
    "triviaqa_mitigation": MITIGATION_REPORT,
    "latency": LATENCY_REPORT,
    "original_v2_reference_only": ORIGINAL_V2_REFERENCE,
}

final_json_path = os.path.join(DIRS["results"], "final_experiment_report.json")
with open(final_json_path, "w", encoding="utf-8") as f:
    json.dump(FINAL_REPORT, f, indent=2, default=float)

# Table 1: In-Domain 5-Fold Group CV + Historical Reference Comparison
t_id = []
for name in IN_DOMAIN_DATASETS:
    r = IN_DOMAIN_RESULTS[name]; f = r["fusion"]; s = r["single_sensor_trained"]
    ref = ORIGINAL_V2_REFERENCE.get(name)
    ref_str = f"{ref['fusion_auroc']:.4f} (n={ref['n_rows']})" if ref else "New Benchmark"
    delta   = f"{f['auroc'] - ref['fusion_auroc']:+.4f}" if ref else "—"
    t_id.append([name, r["n_rows"], f"{f['auroc']:.4f} [{f['auroc_ci95'][0]:.3f},{f['auroc_ci95'][1]:.3f}]",
                 f"{s['ISP']['auroc']:.4f}", f"{s['AAS']['auroc']:.4f}", f"{s['EEM']['auroc']:.4f}", f"{s['DST']['auroc']:.4f}",
                 f"{f['auprc']:.4f}", f"{f['f1']:.4f}", f"{f['ece']:.4f}", ref_str, delta])

print("\n=== TABLE 1: IN-DOMAIN 5-FOLD GROUP-PRESERVING CV (NEWLY COMPUTED) ===")
print(tabulate(t_id, headers=["Dataset", "Rows", "Fusion AUROC [95% CI]", "ISP", "AAS", "EEM", "DST", "AUPRC", "F1", "ECE", "Orig v2 Ref", "Delta"], tablefmt="grid"))

# Table 2: Strictly Unseen Zero-Shot Transfer
t_zs = []
for name in UNSEEN_ZEROSHOT_DATASETS:
    r = UNSEEN_RESULTS[name]; f = r["fusion"]; v = r["variants"]; m = f["at_tau_warning"]
    t_zs.append([name, r["benchmark_type"], r["n_rows"], f"{f['auroc']:.4f} [{f['auroc_ci95'][0]:.3f},{f['auroc_ci95'][1]:.3f}]",
                 f"{v['ISP_only']['auroc']:.4f}", f"{v['without_ISP']['auroc']:.4f}",
                 f"{f['auprc']:.4f}", f"{m['f1']:.4f}", f"{m['precision']:.4f}", f"{m['recall']:.4f}", f"{f['ece']:.4f}"])

print("\n=== TABLE 2: STRICTLY UNSEEN ZERO-SHOT EVALUATION (0 SAMPLES IN TRAINING/CALIBRATION) ===")
print(tabulate(t_zs, headers=["Dataset", "Type", "Rows", "Fusion AUROC [95% CI]", "ISP Only", "AAS+EEM+DST", "AUPRC", "F1@tw", "Prec@tw", "Rec@tw", "ECE"], tablefmt="grid"))

# Strict Final Verification Audit (cannot falsely report success)
required_stages = [
    "embed_original_src",
    "load_and_validate_10_datasets",
    "load_frozen_tinyllama",
    "feature_extraction_9_datasets",
    "in_domain_5fold_cv_and_ablations",
    "unseen_zeroshot_4_datasets",
    "triviaqa_mitigation",
    "latency_profiling",
]
missing_or_failed = [s for s in required_stages if not str(AUDIT["stages"].get(s, "")).startswith("COMPLETED")]
all_9_extracted   = all(name in FEATURES and FEATURES[name][0].shape[0] > 0 for name in IN_DOMAIN_DATASETS + UNSEEN_ZEROSHOT_DATASETS)
all_files_present = all(os.path.exists(p) for p in [
    final_json_path,
    os.path.join(DIRS["results"], "dataset_manifest.json"),
    os.path.join(DIRS["results"], "mitigation_report.json"),
    os.path.join(DIRS["results"], "real_latency.json"),
    os.path.join(DIRS["models"], "trueguard_pooled_in_domain.joblib"),
    os.path.join(DIRS["models"], "trueguard_triviaqa_mitigation.joblib"),
])

print("\n=========================================================================")
print("=== STRICT FINAL EXECUTION AUDIT                                      ===")
print("=========================================================================")
for s in required_stages:
    print(f"  {s:36s} : {AUDIT['stages'].get(s, 'MISSING')}")
print(f"  All 9 feature matrices verified      : {all_9_extracted}")
print(f"  All core artifacts saved to Drive    : {all_files_present}")
print(f"  Master JSON report                   : {final_json_path} ({os.path.getsize(final_json_path):,} bytes)")
print(f"  Total figures generated              : {len(os.listdir(DIRS['figures']))}")

if not missing_or_failed and all_9_extracted and all_files_present:
    print("\nVERDICT: COMPLETE AND SCIENTIFICALLY VERIFIED — All 10 datasets, sensors, ablations, zero-shot transfers, mitigation, and latency profiling completed.")
else:
    print(f"\nVERDICT: INCOMPLETE / FAILED — Missing or failed stages: {missing_or_failed}")
    raise RuntimeError(f"Experiment audit failed: {missing_or_failed}")
